## Library

In [1]:
pip install xgboost scikit-learn matplotlib pandas joblib tqdm

Note: you may need to restart the kernel to use updated packages.


In [2]:
pip install gplearn

Note: you may need to restart the kernel to use updated packages.


In [ ]:
pip install pysr --upgrade 

In [ ]:
pip install PyQt5 matplotlib pandas joblib scipy numpy

In [1]:
pip install pysr

Note: you may need to restart the kernel to use updated packages.


In [2]:
pip install PySide6 matplotlib pandas numpy joblib scipy


Note: you may need to restart the kernel to use updated packages.


In [3]:
pip install PyQt5 matplotlib pandas joblib scipy numpy

Note: you may need to restart the kernel to use updated packages.


In [4]:
pip install PyQt5 matplotlib pandas

Note: you may need to restart the kernel to use updated packages.


In [3]:
pip install tqdm

Note: you may need to restart the kernel to use updated packages.


In [4]:
pip install scikit-learn matplotlib numpy


Note: you may need to restart the kernel to use updated packages.


In [5]:
pip install gradio scikit-learn joblib numpy scipy

Note: you may need to restart the kernel to use updated packages.


In [6]:
pip install gradio scikit-learn joblib numpy pandas matplotlib


Note: you may need to restart the kernel to use updated packages.


In [7]:
pip install xgboost

Note: you may need to restart the kernel to use updated packages.


In [8]:
!pip install gradio scikit-learn joblib numpy scipy --user


In [9]:
!pip install gradio scikit-learn joblib numpy pandas matplotlib --user


In [10]:
!pip install gradio==3.50.2 --user


##  Preparing Data
Before training the surrogate models, we create empty Excel templates for all
combinations of lattice structure and orientation. These templates are then
filled with simulation or experimental results.

In [1]:
import os
import pandas as pd

# ------------------------------
# ⚙️ Setting
# ------------------------------

DATA_DIR = "data"   
os.makedirs(DATA_DIR, exist_ok=True)

STRUCTURES = ["Reentrant", "Neovius", "Gyroid", "Diamond", "SchwarzP"]
ORIENTATIONS = ["UVW", "UWV", "VUW", "VWU", "WUV", "WVU"]

COLUMNS = ["Thickness", "Poisson_X", "Poisson_Y", "Elastic_Modulus", "Volume_Fraction"]

# ------------------------------
# 🏗 Create empity excel files
# ------------------------------

print("🔄 Creating excel files ...")

for structure in STRUCTURES:
    for orient in ORIENTATIONS:
        # name file: Reentrant_UVW.xlsx
        filename = f"{structure}_{orient}.xlsx"
        filepath = os.path.join(DATA_DIR, filename)
        
        # DataFrame 
        df = pd.DataFrame(columns=COLUMNS)
        
        # Save in excel format
        df.to_excel(filepath, index=False, sheet_name="Data")
        
        print(f"✅ {filepath} Created.")

print("\n🎉 All excel files created successfully!")
print(f"📁 Please enter data in empity files '{DATA_DIR}'.")
print("➡️ After entering data go to next step.")

🔄 در حال ایجاد فایل‌های اکسل خالی...
✅ data\Reentrant_UVW.xlsx ایجاد شد.
✅ data\Reentrant_UWV.xlsx ایجاد شد.
✅ data\Reentrant_VUW.xlsx ایجاد شد.
✅ data\Reentrant_VWU.xlsx ایجاد شد.
✅ data\Reentrant_WUV.xlsx ایجاد شد.
✅ data\Reentrant_WVU.xlsx ایجاد شد.
✅ data\Neovius_UVW.xlsx ایجاد شد.
✅ data\Neovius_UWV.xlsx ایجاد شد.
✅ data\Neovius_VUW.xlsx ایجاد شد.
✅ data\Neovius_VWU.xlsx ایجاد شد.
✅ data\Neovius_WUV.xlsx ایجاد شد.
✅ data\Neovius_WVU.xlsx ایجاد شد.
✅ data\Gyroid_UVW.xlsx ایجاد شد.
✅ data\Gyroid_UWV.xlsx ایجاد شد.
✅ data\Gyroid_VUW.xlsx ایجاد شد.
✅ data\Gyroid_VWU.xlsx ایجاد شد.
✅ data\Gyroid_WUV.xlsx ایجاد شد.
✅ data\Gyroid_WVU.xlsx ایجاد شد.
✅ data\Diamond_UVW.xlsx ایجاد شد.
✅ data\Diamond_UWV.xlsx ایجاد شد.
✅ data\Diamond_VUW.xlsx ایجاد شد.
✅ data\Diamond_VWU.xlsx ایجاد شد.
✅ data\Diamond_WUV.xlsx ایجاد شد.
✅ data\Diamond_WVU.xlsx ایجاد شد.
✅ data\SchwarzP_UVW.xlsx ایجاد شد.
✅ data\SchwarzP_UWV.xlsx ایجاد شد.
✅ data\SchwarzP_VUW.xlsx ایجاد شد.
✅ data\SchwarzP_VWU.xlsx ایجاد شد.
✅

##  Reading Data
This Python script automatically reads, validates, cleans, and converts multiple Excel datasets into structured CSV files suitable for training datasets. First, it defines two directories: one called data where the Excel files are expected to be located, and another called training_datasets where the processed CSV files will be saved. The script checks whether the data folder exists and whether it contains any .xlsx files; if not, it raises an error. It then scans all Excel files in the folder and processes them one by one. The filename is parsed to extract two pieces of information: the structure and the orientation, where the orientation is assumed to be the last part of the filename and the structure is everything before it. For each Excel file, the script searches through its sheets and selects the first sheet that contains valid data. Since column names in different files may vary, the script performs flexible matching to map similar column names to a standardized set of expected columns: Thickness, Poisson_X, Poisson_Y, Elastic_Modulus, and Volume_Fraction. It then verifies that these required columns exist. Afterward, it keeps only these columns, converts their values to numeric format (replacing invalid values with NaN), and removes rows only if all values in the row are missing. The cleaned dataset is stored in a dictionary using the combination of structure and orientation as a key. Finally, each processed dataset is saved as a CSV file with a timestamp in its filename inside the training_datasets folder, ensuring the data is preserved exactly without modification or index columns. At the end of the execution, the script prints a summary report showing how many datasets were prepared, where the CSV files were saved, and how many rows each dataset contains, indicating that the data preparation step has been successfully completed and the datasets are ready for the next stage such as plotting or further analysis.

In [1]:
import os
import pandas as pd
from pathlib import Path

# ------------------------------
# ⚙️ Setting
# ------------------------------

DATA_DIR = "data"             
SAVE_DIR = "training_datasets"  
os.makedirs(SAVE_DIR, exist_ok=True)

# Expected columns — only for initial validation
EXPECTED_COLUMNS = ["Thickness", "Poisson_X", "Poisson_Y", "Elastic_Modulus", "Volume_Fraction"]

# ------------------------------
# 📂 List all Excel files in the data folder
# ------------------------------

data_dir_path = Path(DATA_DIR)
if not data_dir_path.exists():
    raise FileNotFoundError(f"❌ Folder '{DATA_DIR}' not found. Please place the Excel files in this folder first.")

excel_files = list(data_dir_path.glob("*.xlsx"))
if not excel_files:
    raise FileNotFoundError(f"❌ No Excel files were found in the '{DATA_DIR}' folder.")

print(f"🔍 {len(excel_files)} Excel files found in the '{DATA_DIR}' folder. Reading...")

# ------------------------------
# 🏗 Dictionary to store all datasets
# key: (structure, orientation), value: DataFrame
# ------------------------------

datasets = {}
file_paths = {}  # To store output CSV file paths

# ------------------------------
# 📖 Reading and processing each file
# ------------------------------

for file_path in excel_files:
    try:
        # --- Parse filename with support for multi‑part names ---
        stem = file_path.stem  # e.g., Reentrant_Honeycomb_UVW
        parts = stem.split('_')
        
        if len(parts) < 2:
            print(f"⚠️  Invalid filename (format: structure_orientation): {file_path.name}")
            continue
        
        # Orientation is always the last part
        orient = parts[-1]
        
        # Structure = all parts except the last one
        structure = "_".join(parts[:-1])
        
        # --- Read the first sheet that contains data ---
        excel_file = pd.ExcelFile(file_path, engine='openpyxl')
        df = None
        for sheet_name in excel_file.sheet_names:
            temp_df = pd.read_excel(excel_file, sheet_name=sheet_name)
            if not temp_df.empty and not temp_df.iloc[0].isnull().all():  # If the first row is not empty
                df = temp_df
                print(f"📄 {file_path.name} -> Sheet '{sheet_name}' selected.")
                break
        
        if df is None:
            print(f"⚠️  {file_path.name} -> No sheet with valid data was found.")
            continue

        # --- Match and correct column names (flexible) ---
        column_mapping = {}
        for col in df.columns:
            c = str(col).strip().lower()
            if "thick" in c:
                column_mapping[col] = "Thickness"
            elif ("poisson" in c or "nu" in c) and "x" in c:
                column_mapping[col] = "Poisson_X"
            elif ("poisson" in c or "nu" in c) and "y" in c:
                column_mapping[col] = "Poisson_Y"
            elif any(kw in c for kw in ["elastic", "modul", "young"]):
                column_mapping[col] = "Elastic_Modulus"
            elif any(kw in c for kw in ["volume", "vol", "fraction"]):
                column_mapping[col] = "Volume_Fraction"

        df = df.rename(columns=column_mapping)

        # --- Ensure required columns exist ---
        missing_cols = [col for col in EXPECTED_COLUMNS if col not in df.columns]
        if missing_cols:
            print(f"❌ {file_path.name} -> Required column(s) not found: {missing_cols}")
            continue

        # --- Select only required columns without dropping rows ---
        df_selected = df[EXPECTED_COLUMNS].copy()

        # --- Convert to numeric (do not drop rows — convert invalid values to NaN) ---
        for col in EXPECTED_COLUMNS:
            df_selected[col] = pd.to_numeric(df_selected[col], errors='coerce')

        # --- Remove rows where *all* values are NaN ---
        df_clean = df_selected.dropna(how='all').reset_index(drop=True)

        if df_clean.empty:
            print(f"⚠️  {file_path.name} -> No data remained after cleaning.")
            continue

        # --- Store in dictionary ---
        datasets[(structure, orient)] = df_clean
        print(f"✅ {structure}_{orient} ← {file_path.name} | {len(df_clean)} rows of data")

    except Exception as e:
        print(f"❌ Error reading {file_path.name}: {str(e)}")
        continue

# ------------------------------
# 💾 Save datasets as CSV — exactly and completely
# ------------------------------

timestamp = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")

for (structure, orient), df in datasets.items():
    filename = f"{structure.lower()}_{orient}_train_{timestamp}.csv"
    filepath = os.path.join(SAVE_DIR, filename)
    
    # Save full data without index and without modification
    df.to_csv(filepath, index=False)
    file_paths[f"{structure}_{orient}"] = filepath

# ------------------------------
# ✅ Final report
# ------------------------------

print("\n" + "="*60)
print("✅ Step 1 completed successfully!")
print(f"📁 Data prepared for {len(datasets)} combinations (structure, orientation).")
print(f"💾 CSV files saved in '{SAVE_DIR}'.\n")

for name, path in file_paths.items():
    df_temp = pd.read_csv(path)
    print(f"📄 {name} → {path} | {len(df_temp)} rows")

print("\n💡 You can now proceed to the next step (plotting graphs).")


🔍 30 Excel files found in the 'data' folder. Reading...
📄 Diamond_UVW.xlsx -> Sheet 'Data' selected.
✅ Diamond_UVW ← Diamond_UVW.xlsx | 35 rows of data
📄 Diamond_UWV.xlsx -> Sheet 'Data' selected.
✅ Diamond_UWV ← Diamond_UWV.xlsx | 35 rows of data
📄 Diamond_VUW.xlsx -> Sheet 'Data' selected.
✅ Diamond_VUW ← Diamond_VUW.xlsx | 35 rows of data
📄 Diamond_VWU.xlsx -> Sheet 'Data' selected.
✅ Diamond_VWU ← Diamond_VWU.xlsx | 35 rows of data
📄 Diamond_WUV.xlsx -> Sheet 'Data' selected.
✅ Diamond_WUV ← Diamond_WUV.xlsx | 35 rows of data
📄 Diamond_WVU.xlsx -> Sheet 'Data' selected.
✅ Diamond_WVU ← Diamond_WVU.xlsx | 35 rows of data
📄 Gyroid_UVW.xlsx -> Sheet 'Data' selected.
✅ Gyroid_UVW ← Gyroid_UVW.xlsx | 42 rows of data
📄 Gyroid_UWV.xlsx -> Sheet 'Data' selected.
✅ Gyroid_UWV ← Gyroid_UWV.xlsx | 42 rows of data
📄 Gyroid_VUW.xlsx -> Sheet 'Data' selected.
✅ Gyroid_VUW ← Gyroid_VUW.xlsx | 42 rows of data
📄 Gyroid_VWU.xlsx -> Sheet 'Data' selected.
✅ Gyroid_VWU ← Gyroid_VWU.xlsx | 42 rows of d

## Drawing Graph 
This Python script creates an interactive desktop application for visualizing and comparing mechanical property datasets using overlay plots. It loads all CSV files from the training_datasets directory, standardizes column names, and converts the Elastic Modulus into a normalized property called Relative Elastic Modulus. During loading, the script detects relevant columns such as Thickness, Poisson’s ratios (X and Y), Elastic Modulus, and Volume Fraction, converts them to numeric values, removes invalid rows, and sorts the data by thickness. The graphical interface is built using PyQt5 and includes controls that allow the user to choose between two plotting modes: Dataset Mode, where specific datasets can be manually selected, and Structure Mode, where datasets are automatically filtered based on a chosen structure and orientation. Users can select which property appears on the X‑axis and Y‑axis, choose line styles, enable or disable grid lines, and overlay multiple datasets in a single plot for comparison. The plotting itself is performed using Matplotlib, embedded directly inside the PyQt window with a navigation toolbar for zooming and interaction. Each dataset is displayed with different colors and markers to make comparisons clear. The application also includes the ability to export the generated plot to image or PDF formats, allowing users to save the visualization for reports or further analysis.

In [ ]:
# ============================================================
# Advanced Overlay Plot
# ============================================================

import sys
import os
import glob
import pandas as pd
import matplotlib
matplotlib.use("Qt5Agg")
import matplotlib.pyplot as plt
from matplotlib.backends.backend_qt5agg import FigureCanvasQTAgg as FigureCanvas
from matplotlib.backends.backend_qt5agg import NavigationToolbar2QT as NavigationToolbar

from PyQt5.QtWidgets import (
    QApplication, QMainWindow, QWidget, QVBoxLayout, QHBoxLayout,
    QLabel, QPushButton, QComboBox, QListWidget, QMessageBox,
    QAbstractItemView, QRadioButton, QButtonGroup, QFileDialog, QCheckBox
)
from PyQt5.QtCore import Qt

# ============================================================
# SETTINGS
# ============================================================
DATA_DIR = "training_datasets"

STRUCTURES = ["Reentrant", "Neovius", "Gyroid", "Diamond", "SchwarzP"]
ORIENTATIONS = ["UVW", "UWV", "VUW", "VWU", "WUV", "WVU"]
# Property list with Relative Elastic Modulus
PROPERTIES = ["Poisson_X", "Poisson_Y", "Relative_Elastic_Modulus", "Volume_Fraction", "Thickness"]
LINE_STYLES = ["Solid", "Dashed", "Dotted"]
COLOR_MAP = plt.cm.tab10.colors
MARKERS = ["o", "s", "^", "D", "v", "P", "*", "X"]

# Display names for axis labels
PROPERTY_DISPLAY_NAMES = {
    "Poisson_X": "Poisson's Ratio (X)",
    "Poisson_Y": "Poisson's Ratio (Y)",
    "Relative_Elastic_Modulus": "Relative Elastic Modulus (E*)",
    "Volume_Fraction": "Volume Fraction",
    "Thickness": "Thickness"
}

# ============================================================
# LOAD ALL CSV FILES — WITH E/1800 NORMALIZATION
# ============================================================
def load_all_csv():
    csv_paths = glob.glob(os.path.join(DATA_DIR, "*.csv"))
    data = {}
    for path in csv_paths:
        try:
            df = pd.read_csv(path)
            # Detect thickness column
            if "Thickness" not in df.columns:
                for col in df.columns:
                    if "thick" in col.lower():
                        df.rename(columns={col: "Thickness"}, inplace=True)
                        break
            
            # Rename and normalize Elastic Modulus → Relative_Elastic_Modulus 
            rename_map = {}
            elastic_col_found = None
            for col in df.columns:
                c = col.lower()
                if "poisson" in c and "x" in c:
                    rename_map[col] = "Poisson_X"
                elif "poisson" in c and "y" in c:
                    rename_map[col] = "Poisson_Y"
                elif "elastic" in c or "young" in c or "modulus" in c:
                    elastic_col_found = col  # Store original column name
                elif "volume" in c or "fraction" in c:
                    rename_map[col] = "Volume_Fraction"
            
            # Apply renaming for non-elastic columns
            df.rename(columns=rename_map, inplace=True)
            
    
            if elastic_col_found:
                df["Relative_Elastic_Modulus"] = df[elastic_col_found] / 1
            else:
                # Skip if no elastic modulus found
                print(f"Skipping {path}: No Elastic Modulus column found")
                continue
            
            # Ensure required columns exist
            required_cols = ["Poisson_X", "Poisson_Y", "Relative_Elastic_Modulus", "Volume_Fraction", "Thickness"]
            if not all(col in df.columns for col in required_cols):
                print(f"Skipping {path}: Missing required columns")
                continue
                
            df = df[required_cols]
            df = df.apply(pd.to_numeric, errors="coerce")
            df.dropna(inplace=True)
            
            if df.empty:
                continue
                
            key = os.path.basename(path)
            data[key] = df.sort_values("Thickness")
            
        except Exception as e:
            print(f"Skipping {path}: {e}")
    
    if not data:
        raise ValueError("No valid CSV files found in the directory.")
    return data

# ============================================================
# MAIN WINDOW
# ============================================================
class MainWindow(QMainWindow):
    def __init__(self, data):
        super().__init__()
        self.all_data = data
        self.filtered_data = {}
        self.initUI()

    def initUI(self):
        self.setWindowTitle("Overlay Plot — Relative Elastic Modulus (E/1800)")
        self.setGeometry(100, 50, 1400, 800)

        central = QWidget()
        self.setCentralWidget(central)
        layout = QHBoxLayout(central)

        # ---------------- Left Panel ----------------
        left = QVBoxLayout()

        # Mode selection
        left.addWidget(QLabel("Select Mode:"))
        self.dataset_mode_radio = QRadioButton("Dataset Mode")
        self.structure_mode_radio = QRadioButton("Structure Mode")
        self.dataset_mode_radio.setChecked(True)
        mode_group = QButtonGroup()
        mode_group.addButton(self.dataset_mode_radio)
        mode_group.addButton(self.structure_mode_radio)
        left.addWidget(self.dataset_mode_radio)
        left.addWidget(self.structure_mode_radio)

        # Structure selection
        left.addWidget(QLabel("Select Structure:"))
        self.struct_combo = QComboBox()
        self.struct_combo.addItems(STRUCTURES)
        self.struct_combo.currentIndexChanged.connect(self.update_filtered_data)
        left.addWidget(self.struct_combo)

        # Orientation selection
        left.addWidget(QLabel("Select Orientation:"))
        self.orient_combo = QComboBox()
        self.orient_combo.addItems(ORIENTATIONS)
        self.orient_combo.currentIndexChanged.connect(self.update_filtered_data)
        left.addWidget(self.orient_combo)

        # X-axis selection
        left.addWidget(QLabel("X-axis:"))
        self.x_combo = QComboBox()
        self.x_combo.addItems(PROPERTIES)
        left.addWidget(self.x_combo)

        # Y-axis selection (exclude Thickness)
        left.addWidget(QLabel("Y-axis:"))
        y_properties = [prop for prop in PROPERTIES if prop != "Thickness"]
        self.y_combo = QComboBox()
        self.y_combo.addItems(y_properties)
        left.addWidget(self.y_combo)

        # Line Style
        left.addWidget(QLabel("Line Style:"))
        self.style_combo = QComboBox()
        self.style_combo.addItems(LINE_STYLES)
        left.addWidget(self.style_combo)

        # Grid option
        self.grid_checkbox = QCheckBox("Show Grid")
        self.grid_checkbox.setChecked(True)
        left.addWidget(self.grid_checkbox)

        # Dataset list
        left.addWidget(QLabel("Datasets:"))
        self.dataset_list = QListWidget()
        self.dataset_list.setSelectionMode(QAbstractItemView.ExtendedSelection)
        self.dataset_list.addItems(list(self.all_data.keys()))
        left.addWidget(self.dataset_list)

        # Buttons
        plot_btn = QPushButton("Plot")
        plot_btn.clicked.connect(self.plot_selected)
        left.addWidget(plot_btn)

        save_btn = QPushButton("Save Plot")
        save_btn.clicked.connect(self.save_plot)
        left.addWidget(save_btn)

        left.addStretch()
        layout.addLayout(left, 1)

        # ---------------- Right Panel ----------------
        right = QVBoxLayout()
        self.canvas = FigureCanvas(plt.Figure())
        self.toolbar = NavigationToolbar(self.canvas, self)
        right.addWidget(self.toolbar)
        right.addWidget(self.canvas)
        layout.addLayout(right, 3)

        self.update_filtered_data()

    def update_filtered_data(self):
        structure = self.struct_combo.currentText().lower()
        orientation = self.orient_combo.currentText().lower()
        self.filtered_data = {}
        for key, df in self.all_data.items():
            if key.lower().startswith(f"{structure}_{orientation}"):
                self.filtered_data[key] = df

        self.dataset_list.clearSelection()
        if self.structure_mode_radio.isChecked():
            for i in range(self.dataset_list.count()):
                item = self.dataset_list.item(i)
                # ✅ FIXED: Was 'self.filtered_' - now 'self.filtered_data'
                if item.text() in self.filtered_data:
                    item.setSelected(True)

    def plot_selected(self):
        mode_dataset = self.dataset_mode_radio.isChecked()
        x_var = self.x_combo.currentText()
        y_var = self.y_combo.currentText()
        
        if x_var == y_var:
            QMessageBox.warning(self, "Error", "X-axis and Y-axis cannot be the same")
            return

        line_style_str = self.style_combo.currentText()
        linestyle_dict = {"Solid": "-", "Dashed": "--", "Dotted": ":"}
        linestyle = linestyle_dict.get(line_style_str, "-")

        fig = self.canvas.figure
        fig.clear()
        ax = fig.add_subplot(111)

        colors = COLOR_MAP
        markers = MARKERS
        count = 0

        if mode_dataset:
            selected_keys = [item.text() for item in self.dataset_list.selectedItems()]
            if not selected_keys:
                QMessageBox.warning(self, "Select", "Please select at least one dataset")
                return
            for key in selected_keys:
                df = self.all_data[key]
                if x_var not in df.columns or y_var not in df.columns:
                    continue
                ax.plot(df[x_var], df[y_var],
                        marker='o', linestyle=linestyle,
                        color=colors[count % len(colors)],
                        linewidth=2, label=key)
                count += 1
        else:
            # ✅ FIXED: Was 'if not self.filtered_' - now 'if not self.filtered_data'
            if not self.filtered_data:
                QMessageBox.warning(self, "No Data", "No datasets for this Structure & Orientation")
                return
            for key, df in self.filtered_data.items():
                if x_var not in df.columns or y_var not in df.columns:
                    continue
                marker = markers[count % len(markers)]
                ax.plot(df[x_var], df[y_var],
                        marker=marker, linestyle=linestyle,
                        color=colors[count % len(colors)],
                        linewidth=2, label=key)
                count += 1

        x_label = PROPERTY_DISPLAY_NAMES.get(x_var, x_var)
        y_label = PROPERTY_DISPLAY_NAMES.get(y_var, y_var)
        ax.set_xlabel(x_label)
        ax.set_ylabel(y_label)
        ax.set_title(f"{y_label} vs {x_label}")
        ax.grid(self.grid_checkbox.isChecked())
        if count > 1:
            ax.legend()
        self.canvas.draw()

    def save_plot(self):
        if self.canvas.figure is None:
            QMessageBox.warning(self, "Save", "Nothing to save yet.")
            return
        file_path, _ = QFileDialog.getSaveFileName(self, "Save Plot", "",
                                                   "PNG (*.png);;PDF (*.pdf);;All Files (*)")
        if file_path:
            self.canvas.figure.savefig(file_path, dpi=150, bbox_inches='tight')
            QMessageBox.information(self, "Saved", f"Plot saved to:\n{file_path}")

# ============================================================
# RUN APP
# ============================================================
if __name__ == "__main__":
    try:
        data = load_all_csv()
    except Exception as e:
        print("Error loading CSV files:", e)
        sys.exit(1)

    app = QApplication(sys.argv)
    window = MainWindow(data)
    window.show()
    sys.exit(app.exec_())

Detected IPython. Loading juliacall extension. See https://juliapy.github.io/PythonCall.jl/stable/compat/#IPython


C:\Users\IRNO\AppData\Local\Temp\ipykernel_19668\3343347519.py:132: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=9)
C:\Users\IRNO\AppData\Local\Temp\ipykernel_19668\3343347519.py:132: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=9)


##  Teaching and Learning 
This Python script performs the final stage of machine learning model training and validation for predicting mechanical properties of different lattice structures based on their thickness. It first loads all prepared CSV datasets from the training_datasets directory, standardizes the column names, and organizes the data by structure type and orientation. The structures considered are Neovius, Reentrant, Gyroid, Diamond, and SchwarzP, and the target properties to predict are Poisson_X, Poisson_Y, Elastic_Modulus, and Volume_Fraction. The script then trains predictive models for every valid combination of structure, orientation, and target property. Training priority is given to Neovius datasets, which are processed first. Different modeling strategies are applied depending on the structure: for Gyroid Poisson ratios, the script uses a hybrid method combining Symbolic Regression (PySR) to discover an analytical equation and a Gaussian Process Regression (GPR) model to learn the residual errors and improve accuracy; for Diamond structures, it applies the original Gaussian Process Regression configuration with polynomial feature expansion and adaptive cross‑validation; and for the remaining structures, it uses an improved configuration based on Gaussian Process Regression, sometimes combined with a tree-based model such as XGBoost or Gradient Boosting in an ensemble. During training, the script performs cross‑validation (Leave-One-Out, ShuffleSplit, or K‑Fold depending on dataset size) to estimate model performance. Several accuracy metrics are calculated, including RMSE, MAE, R², and MAPE, both for cross‑validation predictions and for the full trained model. After training, each model is saved as a .joblib file in the saved_models directory so it can be reused later for predictions. The script also automatically generates validation plots such as True vs Predicted scatter plots and prediction curves versus thickness, which are stored in a plots folder. Finally, it compiles all performance metrics into a final training report CSV file in the reports directory and prints a summary table showing the average prediction accuracy for each structure and target property, along with the total training time and the number of models saved.

In [1]:
import warnings
warnings.filterwarnings("ignore")

import os, glob, time, math
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
from tqdm import tqdm

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, RBF, RationalQuadratic, WhiteKernel, ConstantKernel as C
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.model_selection import LeaveOneOut, KFold, ShuffleSplit
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

try:
    from xgboost import XGBRegressor
    XGB_AVAILABLE = True
except:
    XGB_AVAILABLE = False

try:
    from pysr import PySRRegressor
    PYSP_AVAILABLE = True
except:
    PYSP_AVAILABLE = False
    print("ℹ️ PySR not installed. Gyroid-Poisson will use fallback.")

# --------------------
# SETTINGS
# --------------------
SEED = 42
np.random.seed(SEED)

DATA_DIR = "training_datasets"
MODEL_DIR = "saved_models"
PLOT_DIR = "plots"
REPORT_DIR = "reports"

os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(PLOT_DIR, exist_ok=True)
os.makedirs(REPORT_DIR, exist_ok=True)


STRUCTURES = ["Neovius", "Reentrant", "Gyroid", "Diamond", "SchwarzP"]
ORIENTATIONS = ["UVW", "UWV", "VUW", "VWU", "WUV", "WVU"]
OUTPUTS = ["Poisson_X", "Poisson_Y", "Elastic_Modulus", "Volume_Fraction"]

# Setting 
POLY_DEGREE = 3
GPR_N_RESTARTS = 12
LOO_MAX = 20
SHUFFLE_THRESHOLD = 50
SHUFFLE_N_SPLITS = 30
K_FOLDS = 5

# --------------------
# Helper Functions
# --------------------
def standardize_columns(df):
    df = df.copy()
    col_map = {}
    for col in df.columns:
        c = str(col).strip().lower()
        if "thick" in c:
            col_map[col] = "Thickness"
        elif ("poisson" in c or "nu" in c) and "x" in c:
            col_map[col] = "Poisson_X"
        elif ("poisson" in c or "nu" in c) and "y" in c:
            col_map[col] = "Poisson_Y"
        elif any(kw in c for kw in ["elastic", "modul", "young"]):
            col_map[col] = "Elastic_Modulus"
        elif any(kw in c for kw in ["volume", "vol", "fraction"]):
            col_map[col] = "Volume_Fraction"
    return df.rename(columns=col_map)

def fname_suffix(output):
    return {"Poisson_X":"px","Poisson_Y":"py","Elastic_Modulus":"em","Volume_Fraction":"vf"}.get(output,"out")

def safe_mask(a, b):
    a, b = np.asarray(a), np.asarray(b)
    return np.isfinite(a) & np.isfinite(b)

def rmse(y_true, y_pred):
    mask = safe_mask(y_true, y_pred)
    return math.sqrt(mean_squared_error(y_true[mask], y_pred[mask])) if mask.any() else np.nan

def mae(y_true, y_pred):
    mask = safe_mask(y_true, y_pred)
    return mean_absolute_error(y_true[mask], y_pred[mask]) if mask.any() else np.nan

def r2s(y_true, y_pred):
    mask = safe_mask(y_true, y_pred)
    return r2_score(y_true[mask], y_pred[mask]) if mask.any() else np.nan

def safe_mape(y_true, y_pred):
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    mask = (y_true != 0) & np.isfinite(y_true) & np.isfinite(y_pred)
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100 if mask.any() else np.nan

def plot_true_vs_pred(y_true, y_pred, title, filename):
    plt.figure(figsize=(6,5))
    mask = safe_mask(y_true, y_pred)
    if mask.any():
        plt.scatter(y_true[mask], y_pred[mask], alpha=0.8, s=50, edgecolor='k')
        mn, mx = min(y_true[mask].min(), y_pred[mask].min()), max(y_true[mask].max(), y_pred[mask].max())
        plt.plot([mn,mx],[mn,mx],'r--', linewidth=1)
    plt.xlabel("True")
    plt.ylabel("Pred")
    plt.title(title)
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(filename, dpi=150, bbox_inches='tight')
    plt.close()

def plot_validation_curve(x, y_true, y_pred, title, filename):
    plt.figure(figsize=(8,4.8))
    mask = safe_mask(y_true, y_pred)
    if mask.any():
        idx = np.argsort(x[mask])
        plt.plot(x[mask][idx], y_true[mask][idx], 'o-', label='True', markersize=6)
        plt.plot(x[mask][idx], y_pred[mask][idx], 's--', label='Pred', markersize=5)
    plt.xlabel("Thickness")
    plt.ylabel(title)
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(filename, dpi=150, bbox_inches='tight')
    plt.close()

# --------------------
# Load Data
# --------------------
print("🔍 Loading data...")
data_dict = {}

for struct in STRUCTURES:
    for orient in ORIENTATIONS:
        pattern = os.path.join(DATA_DIR, f"{struct.lower()}_{orient.lower()}_train_*.csv")
        files = sorted(glob.glob(pattern))
        if not files: continue
        try:
            df = pd.read_csv(files[-1])
            df = standardize_columns(df)
            if "Thickness" not in df.columns: continue
            for out in OUTPUTS:
                if out not in df.columns: df[out] = np.nan
            df = df.dropna(subset=["Thickness"])
            df = df.dropna(subset=OUTPUTS, how='all')
            if len(df) < 3: continue
            data_dict[(struct, orient)] = df.reset_index(drop=True)
        except Exception as e:
            print(f"❌ Error loading {struct}-{orient}: {e}")

if not data_dict:
    raise RuntimeError("❌ No valid data found.")

sorted_keys = sorted(data_dict.keys(), key=lambda k: (k[0] != "Neovius", k))  
valid_combos = [(s, o, data_dict[(s, o)]) for (s, o) in sorted_keys]
print(f"\n🎯 {len(valid_combos)} combinations ready. Training order: Neovius first.")

# --------------------
# TRAINING
# --------------------
records = []
saved_models = []
t0 = time.time()
# =========================================================
# Store all predictions for combined plots
# =========================================================

all_plot_data = {
    "Poisson_X": {"true": [], "pred": [], "struct": []},
    "Poisson_Y": {"true": [], "pred": [], "struct": []},
    "Elastic_Modulus": {"true": [], "pred": [], "struct": []},
    "Volume_Fraction": {"true": [], "pred": [], "struct": []},
}
for (struct, orient, df) in tqdm(valid_combos, desc="Training"):
    X_raw = df[["Thickness"]].values.astype(float)
    thickness_all = X_raw.flatten()

    for out in OUTPUTS:
        y_raw = df[out].values.astype(float)
        mask = np.isfinite(thickness_all) & np.isfinite(y_raw)
        X = X_raw[mask].reshape(-1, 1)
        y = y_raw[mask]
        n = len(y)
        if n < 3: continue

        is_gyroid_poisson = (struct.lower() == "gyroid") and (out in ["Poisson_X", "Poisson_Y"])
        is_diamond = (struct.lower() == "diamond")

        # ============================================================== 
        # 🔹  Gyroid-Poisson: Symbolic + GPR_residual
        # ============================================================== 
        if is_gyroid_poisson and PYSP_AVAILABLE:
            try:
                model_sym = PySRRegressor(
                    niterations=300,
                    binary_operators=["+", "-", "*", "/"],
                    unary_operators=["square", "cube", "log", "exp"],
                    maxsize=30,
                    random_state=SEED,
                    verbosity=0
                )
                model_sym.fit(X, y)
                y_sym = model_sym.predict(X)
                residual = y - y_sym

                X_feat = np.column_stack([X, 1.0/(X+1e-6), np.log(X+1e-6)])
                scaler = StandardScaler()
                Xs = scaler.fit_transform(X_feat)

                kernel = C(1.0) * Matern(length_scale=0.05, nu=1.5) + WhiteKernel(noise_level=1e-6)
                gpr = GaussianProcessRegressor(kernel=kernel, normalize_y=True, random_state=SEED, alpha=1e-12)
                gpr.fit(Xs, residual)
                y_final = y_sym + gpr.predict(Xs)

                loo = LeaveOneOut()
                y_oof = np.full_like(y, np.nan)
                for train_i, test_i in loo.split(X):
                    y_sym_test = model_sym.predict(X[test_i])
                    X_test_feat = np.column_stack([X[test_i], 1.0/(X[test_i]+1e-6), np.log(X[test_i]+1e-6)])
                    X_test_scaled = scaler.transform(X_test_feat)
                    y_res_test = gpr.predict(X_test_scaled)
                    y_oof[test_i] = y_sym_test + y_res_test

                algo_name = "symbolic_gpr"
                bundle = {
                    "structure": struct,
                    "orientation": orient,
                    "target": out,
                    "poly": PolynomialFeatures(degree=1),
                    "scaler": scaler,
                    "gpr": gpr,
                    "symbolic_model": model_sym,
                    "tree": None,
                    "wg": 1.0,
                    "wt": 0.0,
                    "algo": algo_name
                }
            except Exception as e:
                print(f"⚠️ Symbolic failed → fallback: {e}")
                is_gyroid_poisson = False

        # ============================================================== 
        # 🔸 Diamond 
        # ============================================================== 
        if is_diamond:
            poly = PolynomialFeatures(degree=POLY_DEGREE, include_bias=False)
            X_poly = poly.fit_transform(X)
            scaler = StandardScaler()
            Xs = scaler.fit_transform(X_poly)

            kernel = (
                C(1.0, (1e-3,1e3)) * (Matern(length_scale=1.0, length_scale_bounds=(1e-2,1e2), nu=2.5) + RBF(length_scale=1.0)) 
                + RationalQuadratic(length_scale=1.0, alpha=1.0) + WhiteKernel(noise_level=1e-6, noise_level_bounds=(1e-9,1e-1))
            )

            gpr = GaussianProcessRegressor(kernel=kernel, normalize_y=True, n_restarts_optimizer=GPR_N_RESTARTS, random_state=SEED)

            if n <= LOO_MAX:
                cv = LeaveOneOut()
            elif n <= SHUFFLE_THRESHOLD:
                cv = ShuffleSplit(n_splits=SHUFFLE_N_SPLITS, test_size=0.3, random_state=SEED)
            else:
                cv = KFold(n_splits=K_FOLDS, shuffle=True, random_state=SEED)

            y_oof_gpr = np.full_like(y, np.nan)
            for train_idx, test_idx in cv.split(Xs):
                try:
                    gpr_fold = GaussianProcessRegressor(kernel=kernel, normalize_y=True, n_restarts_optimizer=3, random_state=SEED)
                    gpr_fold.fit(Xs[train_idx], y[train_idx])
                    y_oof_gpr[test_idx] = gpr_fold.predict(Xs[test_idx])
                except:
                    try:
                        gpr.fit(Xs[train_idx], y[train_idx])
                        y_oof_gpr[test_idx] = gpr.predict(Xs[test_idx])
                    except:
                        y_oof_gpr[test_idx] = np.nan

            try:
                gpr.fit(Xs, y)
                y_final = gpr.predict(Xs)
                y_oof = y_oof_gpr
                algo_name = "gpr"
            except:
                continue

            bundle = {
                "structure": struct,
                "orientation": orient,
                "target": out,
                "poly": poly,
                "scaler": scaler,
                "gpr": gpr,
                "tree": None,
                "wg": 1.0,
                "wt": 0.0,
                "algo": algo_name
            }

        # ============================================================== 
        # 🔸 Other Structures
        # ============================================================== 
        elif not (is_gyroid_poisson and PYSP_AVAILABLE) and not is_diamond:
            poly = PolynomialFeatures(degree=3, include_bias=False)
            X_poly = poly.fit_transform(X)
            scaler = StandardScaler()
            Xs = scaler.fit_transform(X_poly)

            kernel = (
                C(1.0, (1e-4, 1e4)) *
                (Matern(length_scale=1.0, length_scale_bounds=(1e-3, 10), nu=2.5) + RBF(length_scale=2.0)) +
                RationalQuadratic(length_scale=1.5, alpha=0.5) +
                WhiteKernel(noise_level=1e-8, noise_level_bounds=(1e-12, 1e-2))
            )

            gpr = GaussianProcessRegressor(
                kernel=kernel,
                normalize_y=True,
                n_restarts_optimizer=20,
                random_state=SEED,
                alpha=1e-12
            )

            if n <= 30:
                cv = LeaveOneOut()
            else:
                cv = ShuffleSplit(n_splits=30, test_size=0.2, random_state=SEED)

            ensemble = (struct.lower() == "gyroid") and (out in ["Poisson_X", "Poisson_Y"])
            tree = None
            if ensemble:
                if XGB_AVAILABLE:
                    tree = XGBRegressor(
                        n_estimators=1500, learning_rate=0.008, max_depth=5,
                        subsample=0.8, colsample_bytree=0.8, reg_alpha=2, reg_lambda=2,
                        random_state=SEED, verbosity=0
                    )
                else:
                    tree = GradientBoostingRegressor(
                        n_estimators=1000, learning_rate=0.01, max_depth=5, random_state=SEED
                    )

            y_oof_gpr = np.full_like(y, np.nan)
            y_oof_tree = np.full_like(y, np.nan) if ensemble else None

            for train_idx, test_idx in cv.split(Xs):
                try:
                    gpr_fold = GaussianProcessRegressor(kernel=kernel, normalize_y=True, n_restarts_optimizer=5, random_state=SEED)
                    gpr_fold.fit(Xs[train_idx], y[train_idx])
                    y_oof_gpr[test_idx] = gpr_fold.predict(Xs[test_idx])
                except:
                    try:
                        gpr.fit(Xs[train_idx], y[train_idx])
                        y_oof_gpr[test_idx] = gpr.predict(Xs[test_idx])
                    except:
                        y_oof_gpr[test_idx] = np.nan

                if ensemble:
                    try:
                        tree_fold = tree.__class__(**tree.get_params())
                        tree_fold.set_params(random_state=SEED)
                        tree_fold.fit(Xs[train_idx], y[train_idx])
                        y_oof_tree[test_idx] = tree_fold.predict(Xs[test_idx])
                    except:
                        y_oof_tree[test_idx] = np.nan

            gpr_rmse = rmse(y, y_oof_gpr)
            gpr_r2 = r2s(y, y_oof_gpr)
            tree_rmse = tree_r2 = np.nan
            if ensemble:
                tree_rmse = rmse(y, y_oof_tree)
                tree_r2 = r2s(y, y_oof_tree)

            wg, wt = 1.0, 0.0
            algo_name = "gpr"
            if ensemble and not np.isnan(tree_rmse):
                inv_g = 1.0 / (gpr_rmse + 1e-12)
                inv_t = 1.0 / (tree_rmse + 1e-12)
                wg = inv_g / (inv_g + inv_t)
                wt = inv_t / (inv_g + inv_t)
                algo_name = "ensemble"

            try:
                gpr.fit(Xs, y)
            except:
                continue

            final_tree = None
            if ensemble:
                try:
                    tree.fit(Xs, y)
                    final_tree = tree
                except:
                    final_tree = None
                    wt = 0.0
                    wg = 1.0
                    algo_name = "gpr"

            y_g = gpr.predict(Xs)
            if final_tree is not None:
                y_t = final_tree.predict(Xs)
                y_final = wg * y_g + wt * y_t
            else:
                y_final = y_g

            if ensemble and final_tree is not None:
                y_oof = wg * y_oof_gpr + wt * y_oof_tree
            else:
                y_oof = y_oof_gpr

            bundle = {
                "structure": struct,
                "orientation": orient,
                "target": out,
                "poly": poly,
                "scaler": scaler,
                "gpr": gpr,
                "tree": final_tree,
                "wg": wg,
                "wt": wt,
                "algo": algo_name
            }

        # ========== Calculating Parameters ==========
        rmse_cv = rmse(y, y_oof)
        mae_cv = mae(y, y_oof)
        r2_cv = r2s(y, y_oof)
        mape_cv = safe_mape(y, y_oof)
        rmse_full = rmse(y, y_final)
        mae_full = mae(y, y_final)
        r2_full = r2s(y, y_final)
        mape_full = safe_mape(y, y_final)

        # ========== Save models ==========
        model_name = f"{struct.lower()}_{orient.lower()}_{fname_suffix(out)}_{algo_name}.joblib"
        joblib.dump(bundle, os.path.join(MODEL_DIR, model_name))
        saved_models.append(model_name)

        rec = {
            "Structure": struct,
            "Orientation": orient,
            "Target": out,
            "Algo": algo_name,
            "Samples": n,
            "RMSE_CV": rmse_cv,
            "MAE_CV": mae_cv,
            "R2_CV": r2_cv,
            "MAPE_CV": mape_cv,
            "RMSE_full": rmse_full,
            "MAE_full": mae_full,
            "R2_full": r2_full,
            "MAPE_full": mape_full
        }
        records.append(rec)
        
        # =========================================================
        # Save OOF predictions for combined plots
        # =========================================================
        
        all_plot_data[out]["true"].extend(list(y))
        
        # IMPORTANT:
        # Use y_oof instead of y_final
        # because y_oof is cross-validation prediction
        
        all_plot_data[out]["pred"].extend(list(y_oof))
        
        all_plot_data[out]["struct"].extend(
            [struct] * len(y)
        )
        # ========== Drawing Diagrams ==========
        plot_true_vs_pred(y, y_final, f"{struct} ({orient}) - {out}", 
                          os.path.join(PLOT_DIR, f"true_vs_{struct}_{orient}_{fname_suffix(out)}.png"))
        plot_validation_curve(X.flatten(), y, y_final, out, 
                              os.path.join(PLOT_DIR, f"val_{struct}_{orient}_{fname_suffix(out)}.png"))

        print(
            f"✅ {struct}-{orient}-{out} | {algo_name} | "
            f"R²: {r2_cv:.6f} | "
            f"MAE: {mae_cv:.6f} | "
            f"RMSE: {rmse_cv:.6f} | "
            f"MAPE: {mape_cv:.3f}%"
        )


# --------------------
# Final Report
# --------------------
t1 = time.time()
df_report = pd.DataFrame(records)
csv_report = os.path.join(REPORT_DIR, "final_training_report.csv")
df_report.to_csv(csv_report, index=False)

print(f"\n✅ Training completed in {t1-t0:.1f}s.")
print(f"📊 Report: {csv_report}")
print(f"📁 Models saved: {len(saved_models)}")

summary = df_report.groupby(['Structure', 'Target'])[
    ['R2_CV','MAE_CV','RMSE_CV','MAPE_CV']
].mean().round(6)
print("\n📈 Final Accuracy Summary:")
print(summary)
# =========================================================
# PROFESSIONAL COMBINED TRUE vs PRED PLOTS
# =========================================================

print("\n📊 Generating professional combined plots...")

plt.rcParams.update({
    "font.size": 13,
    "axes.labelsize": 14,
    "axes.titlesize": 15,
    "legend.fontsize": 11,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12
})

# ---------------------------------------------------------
# Colors for structures
# ---------------------------------------------------------

struct_colors = {
    "Neovius":  "#E69F00",
    "Reentrant":"#CC79A7",
    "Gyroid":   "#D55E00",
    "Diamond":  "#009E73",
    "SchwarzP": "#0072B2",
}

# ---------------------------------------------------------
# Marker styles
# ---------------------------------------------------------

struct_markers = {
    "Neovius": "o",
    "Reentrant": "s",
    "Gyroid": "^",
    "Diamond": "D",
    "SchwarzP": "P",
}

# =========================================================
# Loop over outputs
# =========================================================

for out in OUTPUTS:

    data = all_plot_data[out]

    y_true_all = np.array(data["true"], dtype=float)
    y_pred_all = np.array(data["pred"], dtype=float)
    structs_all = np.array(data["struct"])

    mask = safe_mask(y_true_all, y_pred_all)

    y_true_all = y_true_all[mask]
    y_pred_all = y_pred_all[mask]
    structs_all = structs_all[mask]

    if len(y_true_all) == 0:
        print(f"⚠️ No data for {out}")
        continue

    # =====================================================
    # Metrics
    # =====================================================

    r2_total = r2s(y_true_all, y_pred_all)
    mae_total = mae(y_true_all, y_pred_all)
    rmse_total = rmse(y_true_all, y_pred_all)

    # =====================================================
    # Figure
    # =====================================================

    fig, ax = plt.subplots(figsize=(7.2, 7.2))

    for s in STRUCTURES:

        idx = (structs_all == s)

        if not np.any(idx):
            continue

        ax.scatter(
            y_true_all[idx],
            y_pred_all[idx],

            marker=struct_markers[s],
            color=struct_colors[s],

            s=70,
            alpha=0.85,

            edgecolors='black',
            linewidths=0.5,

            label=s
        )

    # =====================================================
    # Ideal line
    # =====================================================

    mn = min(y_true_all.min(), y_pred_all.min())
    mx = max(y_true_all.max(), y_pred_all.max())

    ax.plot(
        [mn, mx],
        [mn, mx],

        linestyle='--',
        color='black',
        linewidth=2.0,

        label='Ideal'
    )

    # =====================================================
    # Labels and title
    # =====================================================

    ax.set_xlabel("True value")
    ax.set_ylabel("Predicted value")

    ax.set_title(f"True vs Predicted — {out}")

    # =====================================================
    # Equal axis
    # =====================================================

    ax.set_aspect('equal', adjustable='box')

    # =====================================================
    # Grid
    # =====================================================

    ax.grid(True, alpha=0.25)

    # =====================================================
    # Metrics box
    # =====================================================

    textstr = '\n'.join((
        rf'$R^2 = {r2_total:.4f}$',
        rf'$MAE = {mae_total:.4f}$',
        rf'$RMSE = {rmse_total:.4f}$'
    ))

    ax.text(
        0.05,
        0.95,
        textstr,

        transform=ax.transAxes,

        fontsize=11,

        verticalalignment='top',

        bbox=dict(
            boxstyle='round',
            facecolor='white',
            alpha=0.85
        )
    )

    # =====================================================
    # Legend
    # =====================================================

    ax.legend(
        frameon=True,
        fancybox=True,
        framealpha=0.95
    )

    # =====================================================
    # Save figure
    # =====================================================

    plt.tight_layout()

    fname = os.path.join(
        PLOT_DIR,
        f"professional_combined_{fname_suffix(out)}.png"
    )

    plt.savefig(
        fname,
        dpi=600,
        bbox_inches='tight'
    )

    plt.close()

    print(f"✅ Saved professional plot: {fname}")

Detected IPython. Loading juliacall extension. See https://juliapy.github.io/PythonCall.jl/stable/compat/#IPython
🔍 Loading data...

🎯 30 combinations ready. Training order: Neovius first.


Training:   0%|          | 0/30 [00:00<?, ?it/s]

✅ Neovius-UVW-Poisson_X | gpr | R²: 0.998979 | MAE: 0.000303 | RMSE: 0.000423 | MAPE: 0.110%
✅ Neovius-UVW-Poisson_Y | gpr | R²: 0.999011 | MAE: 0.000299 | RMSE: 0.000416 | MAPE: 0.108%
✅ Neovius-UVW-Elastic_Modulus | gpr | R²: 0.998794 | MAE: 0.003112 | RMSE: 0.007454 | MAPE: 1.602%


Training:   3%|▎         | 1/30 [01:10<34:05, 70.53s/it]

✅ Neovius-UVW-Volume_Fraction | gpr | R²: 0.997431 | MAE: 0.002636 | RMSE: 0.009052 | MAPE: 0.935%
✅ Neovius-UWV-Poisson_X | gpr | R²: 0.998979 | MAE: 0.000303 | RMSE: 0.000423 | MAPE: 0.110%
✅ Neovius-UWV-Poisson_Y | gpr | R²: 0.999011 | MAE: 0.000299 | RMSE: 0.000416 | MAPE: 0.108%
✅ Neovius-UWV-Elastic_Modulus | gpr | R²: 0.998794 | MAE: 0.003112 | RMSE: 0.007454 | MAPE: 1.602%


...

Training:  97%|█████████▋| 29/30 [40:47<01:52, 112.41s/it]

✅ SchwarzP-WUV-Volume_Fraction | gpr | R²: 0.999994 | MAE: 0.000388 | RMSE: 0.000635 | MAPE: 0.101%
✅ SchwarzP-WVU-Poisson_X | gpr | R²: 0.998995 | MAE: 0.000698 | RMSE: 0.001136 | MAPE: 0.202%
✅ SchwarzP-WVU-Poisson_Y | gpr | R²: 0.999069 | MAE: 0.000713 | RMSE: 0.001094 | MAPE: 0.209%
✅ SchwarzP-WVU-Elastic_Modulus | gpr | R²: 0.999946 | MAE: 0.001198 | RMSE: 0.002149 | MAPE: 2.408%


Training: 100%|██████████| 30/30 [42:47<00:00, 85.59s/it] 

✅ SchwarzP-WVU-Volume_Fraction | gpr | R²: 0.999994 | MAE: 0.000388 | RMSE: 0.000635 | MAPE: 0.101%



✅ Training completed in 2567.6s.
📊 Report: reports\final_training_report.csv
📁 Models saved: 120

📈 Final Accuracy Summary:
                              R2_CV    MAE_CV   RMSE_CV    MAPE_CV
Structure Target                                                  
Diamond   Elastic_Modulus  0.972121  0.019106  0.049580   3.360590
          Poisson_X        0.968932  0.001579  0.003157   0.565188
          Poisson_Y        0.967887  0.001613  0.003208   0.574469
          Volume_Fraction  0.999998  0.000222  0.000303   0.058392
Gyroid    Elastic_Modulus  0.999950  0.000821  0.001771   7.989203
          Poisson_X        0.999843  0.000083  0.000159   0.030494
          Poisson_Y        0.999973  0.000021  0.000039   0.007772
          Volume_Fraction  1.000000  0.000132  0.000172   0.051185
Neovius   Elastic_Modulus  0.998794  0.003112  0.007454   1.602197
          Poisson_X        0.998979  0.000303  0.000423   0.109693
          Poisson_Y        0.999011  0.000299  0.000416   0.107646
    

## Predicting
This code implements Stage 4 of a computational framework for the forward and inverse design of cellular lattice structures using previously trained machine learning models. The main objective of the script is to provide an integrated system that can predict mechanical properties from geometric parameters and perform inverse design to determine structural parameters that achieve desired target properties. At the beginning of the script, directories containing the trained models (saved_models) and training datasets (training_datasets) are defined. The program specifies the lattice structures considered in the study (Reentrant, Neovius, Gyroid, Diamond, and SchwarzP), their possible orientations, and the mechanical properties of interest, namely Poisson’s ratio in the X and Y directions, Elastic Modulus, and Volume Fraction. The script then scans all available training CSV files and computes the valid parameter ranges for each property corresponding to every structure–orientation combination. This step ensures that predictions and optimization procedures are restricted to the physically meaningful ranges represented in the training data, thereby preventing extrapolation beyond the learned domain. In addition, the program computes dynamic normalization scales based on the global range of each property across all datasets, which are later used to normalize errors when comparing different properties with significantly different magnitudes.

Subsequently, the script determines the valid thickness bounds for each structure–orientation pair by analyzing the corresponding training datasets. These bounds define the search domain for the design procedures. The next part of the code is responsible for loading trained machine learning models stored in the model directory. Depending on the specific structure and property, the prediction model may consist of a Gaussian Process Regression (GPR) model, a hybrid Symbolic Regression plus GPR residual model, or a combined ensemble of GPR and tree-based regressors. The function responsible for prediction loads the appropriate model bundle and generates property predictions based on the provided thickness value. Using this mechanism, the script can also predict all mechanical properties simultaneously for a given structural configuration.

The code also introduces a Direct Property Mapping capability, which allows the user to directly determine the relationship between two mechanical properties without manually specifying the thickness. In this process, the program first performs an inverse search to determine the thickness that produces a specified input property value, and then uses this thickness to predict the corresponding output property. This is accomplished through a two-step procedure consisting of an initial grid search across the allowable thickness range followed by a high‑precision optimization using the minimize_scalar algorithm from the SciPy library. This approach ensures that the resulting thickness value produces the closest possible match to the requested property value within the valid domain of the model.

Another major component of the code is the Inverse Design module, which determines the thickness required to achieve a specified target value of a single mechanical property. The algorithm first performs a high‑resolution grid search across the thickness domain to identify candidate solutions that approximate the target value. The best candidates are then refined through local numerical optimization, resulting in a set of valid solutions ranked according to their prediction error. The optimal thickness corresponding to the smallest deviation from the target value is then reported as the final design recommendation.

In addition to single‑target inverse design, the script implements a more advanced Composite Inverse Design framework, which enables simultaneous optimization of multiple mechanical properties. In this mode, the user may activate several target properties and define acceptable tolerance ranges for each one. The algorithm then evaluates all selected combinations of lattice structures and orientations, scanning the allowable thickness range and predicting the associated properties using the trained models. For each candidate thickness, the algorithm calculates normalized errors based on the dynamic property scales, ensuring a fair comparison between properties with different units and magnitudes. Solutions are ranked based on the number of properties that satisfy the specified tolerances and the overall normalized error. The results include the best solution for each structure as well as the global optimal configuration across all evaluated combinations.

Finally, the script constructs an interactive graphical interface using the Gradio framework. This interface allows users to easily interact with the design system through four operational modes: Forward Prediction, Inverse Design, Composite Inverse Design, and Direct Property Mapping. Users can select the lattice structure, orientation, and thickness, or define target mechanical properties and tolerances. The interface also allows users to restrict the search to specific structures or orientations and to control the numerical precision of the results. After executing the analysis, the system displays structured outputs including predicted properties, optimal thickness values, error metrics, and ranked solution tables. When the script is executed, the Gradio application launches in a web browser, providing an accessible tool for data‑driven design and analysis of cellular lattice structures using machine learning models.

In [1]:
# ==============================================================
# 🧊 Stage 4: Forward and Inverse Design with Structure Selection4)
# ==============================================================

import os
import numpy as np
import pandas as pd
import joblib
from scipy.optimize import minimize_scalar
import gradio as gr

# --------------------------
# ⚙️ Settings
# --------------------------
MODEL_DIR = "saved_models"
DATA_DIR = "training_datasets"

STRUCTURES = ["Reentrant", "Neovius", "Gyroid", "Diamond", "SchwarzP"]
ORIENTATIONS = ["UVW", "UWV", "VUW", "VWU", "WUV", "WVU"]
PROPERTIES = ["Poisson_X", "Poisson_Y", "Elastic_Modulus", "Volume_Fraction"]

OUTPUT_TO_KEY = {
    "Poisson_X": "px",
    "Poisson_Y": "py",
    "Elastic_Modulus": "em",
    "Volume_Fraction": "vf",
}

# Default precision setting (applies to ALL numerical values)
DEFAULT_PRECISION = 3

# -----------------------------------------------------------
# 🔍 NEW SECTION: Compute valid parameter ranges from training data
# -----------------------------------------------------------
def compute_valid_ranges():
    """Scan all CSV files and extract min/max for each parameter per (structure, orientation)"""
    ranges = {}
    for structure in STRUCTURES:
        ranges[structure] = {}
        for orientation in ORIENTATIONS:
            struct_pattern = f"{structure.lower()}_{orientation.lower()}_train"
            files = [f for f in os.listdir(DATA_DIR) 
                    if f.lower().endswith(".csv") and struct_pattern in f.lower()]
            
            if not files:
                ranges[structure][orientation] = {prop: None for prop in PROPERTIES}
                continue
            
            files.sort(key=lambda x: os.path.getmtime(os.path.join(DATA_DIR, x)), reverse=True)
            latest_file = files[0]
            
            try:
                df = pd.read_csv(os.path.join(DATA_DIR, latest_file))
                prop_ranges = {}
                
                for prop in PROPERTIES:
                    if prop in df.columns:
                        numeric_vals = pd.to_numeric(df[prop], errors="coerce").dropna()
                        if len(numeric_vals) > 0:
                            prop_ranges[prop] = (float(numeric_vals.min()), float(numeric_vals.max()))
                        else:
                            prop_ranges[prop] = None
                    else:
                        prop_ranges[prop] = None
                
                ranges[structure][orientation] = prop_ranges
                
            except Exception as e:
                print(f"⚠️ Error processing {latest_file}: {e}")
                ranges[structure][orientation] = {prop: None for prop in PROPERTIES}
    
    return ranges

# Precompute valid ranges at startup
VALID_RANGES = compute_valid_ranges()
print("✅ Valid parameter ranges computed from training data")

# -----------------------------------------------------------
# 🔍 Compute dynamic normalization scales from training data
# -----------------------------------------------------------
def compute_dynamic_scales():
    """
    Compute normalization scales based on actual data ranges.
    Scale = (max_value - min_value) for each property across all data.
    This ensures fair contribution regardless of target magnitude.
    """
    scales = {}
    for prop in PROPERTIES:
        all_values = []
        for struct in STRUCTURES:
            for orient in ORIENTATIONS:
                if VALID_RANGES[struct][orient][prop] is not None:
                    min_val, max_val = VALID_RANGES[struct][orient][prop]
                    all_values.extend([min_val, max_val])
        
        if all_values:
            prop_range = float(np.max(all_values) - np.min(all_values))
            scales[prop] = max(prop_range, 1e-6)
        else:
            fallback_scales = {
                "Poisson_X": 0.2,
                "Poisson_Y": 0.2, 
                "Elastic_Modulus": 200.0,
                "Volume_Fraction": 0.5
            }
            scales[prop] = fallback_scales[prop]
            print(f"⚠️ Using fallback scale for {prop}: {scales[prop]}")
    
    return scales

# Compute dynamic scales AFTER VALID_RANGES is available
DYNAMIC_SCALES = compute_dynamic_scales()
print(f"✅ Dynamic normalization scales computed: {DYNAMIC_SCALES}")

# -----------------------------------------------------------
# 🔍 Read thickness bounds — robust to filename variations
# -----------------------------------------------------------
def get_thickness_bounds(structure, orient):
    struct_lower = structure.lower()
    orient_lower = orient.lower()
    target_pattern = f"{struct_lower}_{orient_lower}_train"
    
    files = []
    for f in os.listdir(DATA_DIR):
        if f.lower().endswith(".csv") and target_pattern in f.lower():
            files.append(f)
    
    if not files:
        return None
    
    files.sort(key=lambda x: os.path.getmtime(os.path.join(DATA_DIR, x)), reverse=True)
    latest = files[0]
    
    try:
        df = pd.read_csv(os.path.join(DATA_DIR, latest))
        for col in df.columns:
            if "thick" in col.lower():
                vals = pd.to_numeric(df[col], errors="coerce").dropna().values
                if len(vals) > 0:
                    return float(np.min(vals)), float(np.max(vals))
    except Exception as e:
        print(f"⚠️ Failed to read bounds from {latest}: {e}")
        pass
    return None

# -----------------------------------------------------------
# Model loading & prediction
# -----------------------------------------------------------
def find_model_path(structure, orient, output):
    suffix = OUTPUT_TO_KEY[output]
    prefix = f"{structure.lower()}_{orient.lower()}_{suffix}"
    candidates = []
    for f in os.listdir(MODEL_DIR):
        if f.lower().startswith(prefix.lower()) and f.endswith(".joblib"):
            candidates.append(f)
    return os.path.join(MODEL_DIR, sorted(candidates)[-1]) if candidates else None

def load_model_bundle(structure, orient, output):
    path = find_model_path(structure, orient, output)
    if path is None:
        raise FileNotFoundError(f"Model not found for {structure}-{orient}-{output}.")
    return joblib.load(path)

def predict_property(structure, orient, output, thickness):
    bundle = load_model_bundle(structure, orient, output)
    X = np.array([[float(thickness)]])
    algo = bundle.get("algo", "gpr")
    
    if algo == "symbolic_gpr":
        symbolic_model = bundle["symbolic_model"]
        sym_pred = symbolic_model.predict(X)[0]
        X_feat = np.column_stack([X, 1.0/(X+1e-6), np.log(X+1e-6)])
        X_scaled = bundle["scaler"].transform(X_feat)
        residual_pred = bundle["gpr"].predict(X_scaled)[0]
        return float(sym_pred + residual_pred)
    
    else:
        is_gyroid_poisson = bundle.get("is_gyroid_poisson", False)
        if is_gyroid_poisson:
            X_feat = np.column_stack([
                X,
                1.0 / (X + 1e-6),
                np.log(X + 1e-6),
                np.sqrt(X),
                X ** 0.333
            ])
        else:
            poly = bundle["poly"]
            X_feat = poly.transform(X)
        
        X_scaled = bundle["scaler"].transform(X_feat)
        gpr = bundle["gpr"]
        tree = bundle.get("tree", None)
        wg = bundle.get("wg", 1.0)
        wt = bundle.get("wt", 0.0)
        
        y_gpr = gpr.predict(X_scaled)[0]
        if tree is not None:
            y_tree = tree.predict(X_scaled)[0]
            y_final = wg * y_gpr + wt * y_tree
        else:
            y_final = y_gpr
        return float(y_final)

def predict_all_properties(structure, orient, thickness):
    results = {}
    for prop in PROPERTIES:
        try:
            thick_bounds = get_thickness_bounds(structure, orient)
            if thick_bounds and not (thick_bounds[0] <= thickness <= thick_bounds[1]):
                results[prop] = float('nan')
            else:
                results[prop] = predict_property(structure, orient, prop, thickness)
        except Exception:
            results[prop] = float('nan')
    return results

# -----------------------------------------------------------
# 🔥 Helper function: Round value based on user precision
# -----------------------------------------------------------
def round_value(value, precision):
    """Round value to specified precision while handling NaN/None"""
    if value is None or np.isnan(value):
        return value
    return float(np.round(value, precision))

# -----------------------------------------------------------
# 🔥 NEW: Direct Property Mapping Function
# -----------------------------------------------------------
def direct_property_mapping(structure, orient, input_prop, input_value, output_prop, precision):
    """
    Directly map from input property to output property without thickness intermediate.
    Uses inverse design to find thickness, then predicts output property.
    """
    # Validate input property
    if VALID_RANGES[structure][orient][input_prop] is None:
        raise ValueError(f"No training data available for {input_prop} in {structure}-{orient}.")
    
    min_val, max_val = VALID_RANGES[structure][orient][input_prop]
    if not (min_val <= input_value <= max_val):
        raise ValueError(f"Input {input_value:.{precision}f} outside valid range [{min_val:.{precision}f}, {max_val:.{precision}f}] for {structure}-{orient}-{input_prop}")
    
    # Validate output property
    if VALID_RANGES[structure][orient][output_prop] is None:
        raise ValueError(f"No training data available for {output_prop} in {structure}-{orient}.")
    
    # Step 1: Find thickness that gives the input value (inverse design)
    bounds = get_thickness_bounds(structure, orient)
    if bounds is None:
        raise RuntimeError(f"No training data found for {structure}-{orient}.")
    low, high = float(bounds[0]), float(bounds[1])
    
    # Grid search for initial thickness
    grid_thicks = np.linspace(low, high, 500)
    best_err = float('inf')
    best_thick = None
    
    for t in grid_thicks:
        try:
            pred = predict_property(structure, orient, input_prop, t)
            if min_val <= pred <= max_val:
                err = abs(pred - input_value)
                if err < best_err:
                    best_err = err
                    best_thick = t
                    if err < 1e-6:  # Early termination
                        break
        except:
            continue
    
    if best_thick is None:
        raise RuntimeError(f"Could not find thickness for input {input_prop} = {input_value:.{precision}f}")
    
    # Step 2: High-precision optimization
    def objective(t):
        try:
            pred = predict_property(structure, orient, input_prop, t)
            if min_val <= pred <= max_val:
                return (pred - input_value) ** 2
        except:
            pass
        return 1e10
    
    res = minimize_scalar(
        objective,
        bounds=(low, high),
        method='bounded',
        options={'xatol': 1e-10, 'maxiter': 200}
    )
    
    if res.success:
        t_opt = float(res.x)
    else:
        t_opt = best_thick
    
    # Step 3: Predict output property at this thickness
    output_value = predict_property(structure, orient, output_prop, t_opt)
    
    # Validate output is within bounds
    out_min, out_max = VALID_RANGES[structure][orient][output_prop]
    if not (out_min <= output_value <= out_max):
        raise RuntimeError(f"Predicted output {output_value:.{precision}f} outside valid range [{out_min:.{precision}f}, {out_max:.{precision}f}]")
    
    return t_opt, output_value

# -----------------------------------------------------------
# 🔥 Single-target inverse design WITH UNIFIED PRECISION
# -----------------------------------------------------------
def inverse_design(structure, orient, output, target_value, precision, tol=1e-8, n_grid=800):
    if VALID_RANGES[structure][orient][output] is None:
        raise ValueError(f"No training data available for {output} in {structure}-{orient}.")
    
    min_val, max_val = VALID_RANGES[structure][orient][output]
    if not (min_val <= target_value <= max_val):
        raise ValueError(f"Target {target_value:.6f} outside valid range [{min_val:.6f}, {max_val:.6f}] for {structure}-{orient}-{output}")
    
    bounds = get_thickness_bounds(structure, orient)
    if bounds is None:
        raise RuntimeError(f"No training data found for {structure}-{orient}.")
    low, high = float(bounds[0]), float(bounds[1])
    
    all_valid_solutions = []
    best_err = float('inf')
    best_thick = None
    
    # === HIGH-RESOLUTION GRID SEARCH ===
    grid_thicks = np.linspace(low, high, n_grid)
    
    for t in grid_thicks:
        try:
            pred = predict_property(structure, orient, output, t)
            if not (min_val <= pred <= max_val):
                continue
            err = abs(pred - float(target_value))
            all_valid_solutions.append({
                "thickness": t,
                "predicted": pred,
                "error": err
            })
            if err < best_err:
                best_err = err
                best_thick = t
        except:
            continue
            
    if not all_valid_solutions:
        raise RuntimeError("No valid predictions found.")
    
    # === HIGH-PRECISION OPTIMIZATION AROUND BEST CANDIDATES ===
    all_valid_solutions.sort(key=lambda x: x["error"])
    top_candidates = all_valid_solutions[:5]
    
    refined_solutions = []
    for candidate in top_candidates:
        t0 = candidate["thickness"]
        width = (high - low) / 30
        opt_low = max(low, t0 - width)
        opt_high = min(high, t0 + width)
        if opt_low >= opt_high:
            opt_low, opt_high = low, high
        
        def objective(t):
            try:
                pred = predict_property(structure, orient, output, t)
                if not (min_val <= pred <= max_val):
                    return 1e10
                return (pred - float(target_value)) ** 2
            except:
                return 1e10
        
        res = minimize_scalar(
            objective,
            bounds=(opt_low, opt_high),
            method='bounded',
            options={'xatol': 1e-12, 'maxiter': 300}
        )
        
        if res.success and low <= res.x <= high:
            t_opt = float(res.x)
            actual = predict_property(structure, orient, output, t_opt)
            final_err = abs(actual - float(target_value))
            refined_solutions.append({
                "thickness": t_opt,
                "predicted": actual,
                "error": final_err
            })
    
    # Combine solutions
    all_solutions = all_valid_solutions + refined_solutions
    
    # Remove duplicates based on USER PRECISION (applies to both thickness and predicted values)
    unique_solutions = []
    for sol in all_solutions:
        is_duplicate = False
        rounded_thick = round_value(sol["thickness"], precision)
        rounded_pred = round_value(sol["predicted"], precision)
        for unique_sol in unique_solutions:
            if (abs(rounded_thick - round_value(unique_sol["thickness"], precision)) < 10**(-precision) and
                abs(rounded_pred - round_value(unique_sol["predicted"], precision)) < 10**(-precision)):
                is_duplicate = True
                break
        if not is_duplicate:
            unique_solutions.append(sol)
    
    if not unique_solutions:
        raise RuntimeError("No valid solutions after refinement.")
    
    unique_solutions.sort(key=lambda x: x["error"])
    optimal_solution = unique_solutions[0]
    
    return unique_solutions, optimal_solution, precision

# -----------------------------------------------------------
# -----------------------------------------------------------
# 🔥 COMPOSITE INVERSE DESIGN WITH STRUCTURE SELECTION
# -----------------------------------------------------------
def run_composite_inverse(
    em_active, em_target, em_tol,
    px_active, px_target, px_tol,
    py_active, py_target, py_tol,
    vf_active, vf_target, vf_tol,
    precision,
    selected_structures,
    selected_orientations,
    n_grid=800
):
    targets = {}
    tolerances = {}
    scales = {}
    
    if em_active and em_target is not None:
        targets["Elastic_Modulus"] = float(em_target)
        tolerances["Elastic_Modulus"] = float(em_tol)
        scales["Elastic_Modulus"] = DYNAMIC_SCALES["Elastic_Modulus"]
    if px_active and px_target is not None:
        targets["Poisson_X"] = float(px_target)
        tolerances["Poisson_X"] = float(px_tol)
        scales["Poisson_X"] = DYNAMIC_SCALES["Poisson_X"]
    if py_active and py_target is not None:
        targets["Poisson_Y"] = float(py_target)
        tolerances["Poisson_Y"] = float(py_tol)
        scales["Poisson_Y"] = DYNAMIC_SCALES["Poisson_Y"]
    if vf_active and vf_target is not None:
        targets["Volume_Fraction"] = float(vf_target)
        tolerances["Volume_Fraction"] = float(vf_tol)
        scales["Volume_Fraction"] = DYNAMIC_SCALES["Volume_Fraction"]
    
    if not targets:
        return "⚠️ Please activate at least one target property."

    # Validate selected combinations
    valid_structures = [s for s in selected_structures if s in STRUCTURES]
    valid_orientations = [o for o in selected_orientations if o in ORIENTATIONS]
    
    if not valid_structures or not valid_orientations:
        return "❌ Please select at least one valid structure and one valid orientation."

    all_valid_solutions = []

    # Only search selected combinations
    for struct in valid_structures:
        for orient in valid_orientations:
            try:
                bounds = get_thickness_bounds(struct, orient)
                if bounds is None:
                    continue
                low, high = float(bounds[0]), float(bounds[1])

                grid_thicks = np.linspace(low, high, n_grid)
                
                for t in grid_thicks:
                    predictions = {}
                    normalized_errors = {}
                    in_tolerance = {}
                    valid = True
                    total_norm_error = 0.0
                    
                    for prop in targets:
                        try:
                            pred_val = predict_property(struct, orient, prop, t)
                            if VALID_RANGES[struct][orient][prop] is not None:
                                min_val, max_val = VALID_RANGES[struct][orient][prop]
                                if pred_val < min_val or pred_val > max_val:
                                    valid = False
                                    break
                            
                            target_val = targets[prop]
                            abs_err = abs(pred_val - target_val)
                            predictions[prop] = pred_val
                            normalized_errors[prop] = abs_err / scales[prop]
                            in_tolerance[prop] = (abs_err <= tolerances[prop])
                            total_norm_error += abs_err / scales[prop]
                            
                        except Exception:
                            valid = False
                            break
                    
                    if not valid:
                        continue
                    
                    num_matched = sum(in_tolerance.values())
                    if num_matched == 0:
                        continue
                    
                    avg_norm_error = total_norm_error / len(targets)
                    full_preds = predict_all_properties(struct, orient, t)
                    for prop in targets:
                        full_preds[prop] = predictions[prop]
                    
                    matched_props = [prop for prop in targets if in_tolerance[prop]]
                    all_valid_solutions.append({
                        "structure": struct,
                        "orientation": orient,
                        "thickness": t,
                        "num_matched": num_matched,
                        "matched_props": ", ".join(matched_props),
                        "total_norm_error": avg_norm_error,
                        "predictions": full_preds,
                        "in_tolerance": in_tolerance.copy(),
                        "precision": precision
                    })

                # Optimization logic

            except Exception as e:
                continue

    if not all_valid_solutions:
        return f"❌ No solution found in selected combinations ({len(valid_structures)} structures × {len(valid_orientations)} orientations = {len(valid_structures)*len(valid_orientations)} combos)."

    all_valid_solutions.sort(key=lambda x: (-x["num_matched"], x["total_norm_error"]))
    
    # === OUTPUT WITH UNIFIED PRECISION ===
    txt = "### 🌟 COMPOSITE INVERSE DESIGN RESULTS\n\n"
    
    solutions_by_structure = {}
    for sol in all_valid_solutions:
        struct = sol['structure']
        if struct not in solutions_by_structure:
            solutions_by_structure[struct] = []
        solutions_by_structure[struct].append(sol)
    
    global_best = all_valid_solutions[0]
    
    for struct in valid_structures:  # Only show selected structures
        if struct in solutions_by_structure:
            struct_sols = solutions_by_structure[struct]
            struct_sols.sort(key=lambda x: (-x["num_matched"], x["total_norm_error"]))
            best_for_struct = struct_sols[0]
            
            txt += f"## 🏗️ {struct} ({len(struct_sols)} valid solutions)\n"
            txt += f"| Orientation | Thickness (mm) | Matched Count | Matched Properties | Avg. Norm. Error | Poisson_X | Poisson_Y | Elastic_Modulus | Volume_Fraction |\n"
            txt += f"|-------------|----------------|---------------|--------------------|------------------|-----------|-----------|-----------------|-----------------|\n"
            
            for sol in struct_sols:
                p = sol["predictions"]
                poisson_x = f"{p['Poisson_X']:+.{precision}f}" if not np.isnan(p['Poisson_X']) else "N/A"
                poisson_y = f"{p['Poisson_Y']:+.{precision}f}" if not np.isnan(p['Poisson_Y']) else "N/A"
                elastic = f"{p['Elastic_Modulus']:.{precision}f}" if not np.isnan(p['Elastic_Modulus']) else "N/A"
                vol_frac = f"{p['Volume_Fraction']:.{precision}f}" if not np.isnan(p['Volume_Fraction']) else "N/A"
                thickness_str = f"{sol['thickness']:.{precision}f}"
                
                highlight = " ⭐" if sol == best_for_struct else ""
                txt += f"| {sol['orientation']} | {thickness_str} | {sol['num_matched']} | {sol['matched_props']} | {sol['total_norm_error']:.2e} | {poisson_x} | {poisson_y} | {elastic} | {vol_frac} |{highlight}\n"
            
            txt += f"\n### 🥇 Best for {struct}:\n"
            txt += f"- **Orientation**: {best_for_struct['orientation']}\n"
            txt += f"- **Thickness**: **{best_for_struct['thickness']:.{precision}f} mm**\n"
            txt += f"- **Matched Properties**: **{best_for_struct['matched_props']}**\n"
            txt += f"- **Average Normalized Error**: **{best_for_struct['total_norm_error']:.2e}**\n\n"
        else:
            txt += f"## 🏗️ {struct} (0 valid solutions)\n"
            txt += f"> ⚠️ No solutions found in selected orientations: {', '.join(valid_orientations)}\n\n"
    
    txt += "## 🏆 GLOBAL OPTIMUM ACROSS SELECTED COMBINATIONS\n"
    txt += f"- **Structure**: {global_best['structure']}\n"
    txt += f"- **Orientation**: {global_best['orientation']}\n"
    txt += f"- **Thickness**: **{global_best['thickness']:.{precision}f} mm**\n"
    txt += f"- **Matched Properties**: **{global_best['matched_props']}**\n"
    txt += f"- **Average Normalized Error**: **{global_best['total_norm_error']:.2e}**\n\n"
    
    txt += "### 🔍 Global Best Property Details:\n"
    for prop in targets:
        target_val = targets[prop]
        pred_val = global_best['predictions'].get(prop, np.nan)
        if np.isnan(pred_val):
            txt += f"- ❌ **{prop}**: Prediction failed\n"
        else:
            abs_err = abs(pred_val - target_val)
            tol = tolerances[prop]
            status = "✅" if abs_err <= tol else "❌"
            txt += f"- {status} **{prop}**: Target = {target_val:+.{precision}f} ± {tol:.2e} → Predicted = {pred_val:+.{precision}f} (Error = {abs_err:.2e})\n"
    
    txt += f"\n💡 **Total Valid Solutions**: {len(all_valid_solutions)}"
    txt += f"\n📊 **Structures Searched**: {len(valid_structures)} (selected from {len(STRUCTURES)})"
    txt += f"\n🧭 **Orientations Searched**: {len(valid_orientations)} (selected from {len(ORIENTATIONS)})"
    # ✅ CORRECTED: Fixed variable name
    txt += f"\n🔍 **Total Combinations Searched**: {len(valid_structures)} × {len(valid_orientations)} = {len(valid_structures) * len(valid_orientations)}"
    txt += f"\n🎯 **User Precision Setting**: {precision} decimal places"

    return txt

# -----------------------------------------------------------
# 🎛 Gradio Interface with Structure Selection
# -----------------------------------------------------------
def run_app(mode, structure, orientation, thickness, output_prop, target_val,
            em_active, em_target, em_tol,
            px_active, px_target, px_tol,
            py_active, py_target, py_tol,
            vf_active, vf_target, vf_tol,
            precision,
            selected_structures,
            selected_orientations,
            input_prop_direct=None, input_val_direct=None, output_prop_direct=None):
    try:
        precision = int(precision)
        
        if mode == "Forward":
            if thickness is None or np.isnan(thickness):
                return "⚠️ Please enter a valid thickness."
            bounds = get_thickness_bounds(structure, orientation)
            if bounds is None:
                return f"❌ No training data found for {structure}-{orientation}."
            if not (bounds[0] <= thickness <= bounds[1]):
                return f"⚠️ Thickness must be in [{bounds[0]:.{precision}f}, {bounds[1]:.{precision}f}] mm."
                
            preds = predict_all_properties(structure, orientation, thickness)
            txt = f"### 🔵 Forward Prediction\n**Structure**: {structure} | **Orientation**: {orientation} | **Thickness**: {thickness:.{precision}f} mm\n\n"
            for prop in PROPERTIES:
                val = preds[prop]
                if np.isnan(val):
                    if VALID_RANGES[structure][orientation][prop] is None:
                        txt += f"- {prop}: ❌ No data available\n"
                    else:
                        txt += f"- {prop}: ❌ Prediction out of valid range\n"
                elif "Poisson" in prop:
                    txt += f"- {prop}: **{val:+.{precision}f}**\n"
                elif "Elastic" in prop:
                    txt += f"- {prop}: **{val:.{precision}f}**\n"
                else:
                    txt += f"- {prop}: **{val:.{precision}f}**\n"
            return txt
            
        elif mode == "Inverse":
            if output_prop is None or target_val is None:
                return "⚠️ Please select a target property and value."
            
            if VALID_RANGES[structure][orientation][output_prop] is None:
                return f"❌ No data available for {output_prop} in {structure}-{orientation}."
            
            min_val, max_val = VALID_RANGES[structure][orientation][output_prop]
            if not (min_val <= target_val <= max_val):
                return f"❌ Target {target_val:.{precision}f} outside valid range [{min_val:.{precision}f}, {max_val:.{precision}f}] for {structure}-{orientation}-{output_prop}."
            
            all_solutions, optimal_solution, _ = inverse_design(structure, orientation, output_prop, target_val, precision)
            t_opt = optimal_solution["thickness"]
            actual = optimal_solution["predicted"]
            abs_err = optimal_solution["error"]
            rel_err = (abs_err / abs(float(target_val))) * 100 if target_val != 0 else 0.0
            
            txt = f"### 🔥 Inverse Design\n**Structure**: {structure} | **Orientation**: {orientation}\n"
            txt += f"- Target: **{output_prop} = {float(target_val):+.{precision}f}** (Valid range: [{min_val:.{precision}f}, {max_val:.{precision}f}])\n"
            txt += f"- **Optimal Thickness**: **{t_opt:.{precision}f} mm**\n"
            txt += f"- **Predicted Value**: **{actual:+.{precision}f}**\n"
            txt += f"- **Absolute Error**: **{abs_err:.2e}**\n"
            if target_val != 0:
                txt += f"- **Relative Error**: **{rel_err:.4f}%**\n\n"
            
            txt += f"### 🔍 All Valid Solutions Found ({len(all_solutions)} total):\n"
            txt += "| # | Thickness (mm) | Predicted Value | Absolute Error |\n"
            txt += "|---|----------------|-----------------|----------------|\n"
            
            display_solutions = all_solutions[:10]
            for i, sol in enumerate(display_solutions, 1):
                thickness_val = sol["thickness"]
                pred_val = sol["predicted"]
                error = sol["error"]
                highlight = " ⭐" if (abs(thickness_val - t_opt) < 10**(-precision)) else ""
                txt += f"| {i} | {thickness_val:.{precision}f} | {pred_val:+.{precision}f} | {error:.2e} |{highlight}\n"
            
            if len(all_solutions) > 10:
                txt += f"\n> 💡 Showing top 10 solutions (total: {len(all_solutions)})"
            
            return txt
            
        elif mode == "Direct Property Mapping":
            # New fourth mode
            if input_prop_direct is None or output_prop_direct is None or input_val_direct is None:
                return "⚠️ Please select both input and output properties and provide an input value."
            
            if input_prop_direct == output_prop_direct:
                return "⚠️ Input and output properties cannot be the same."
            
            try:
                t_opt, output_val = direct_property_mapping(
                    structure, orientation, 
                    input_prop_direct, float(input_val_direct), 
                    output_prop_direct, precision
                )
                
                txt = f"### 🔄 Direct Property Mapping\n"
                txt += f"**Structure**: {structure} | **Orientation**: {orientation}\n"
                txt += f"- **Input**: **{input_prop_direct} = {float(input_val_direct):.{precision}f}**\n"
                txt += f"- **Output**: **{output_prop_direct} = {output_val:.{precision}f}**\n"
                txt += f"- **Intermediate Thickness**: **{t_opt:.{precision}f} mm**\n"
                txt += f"\n> 💡 This result was obtained in a single step without manual thickness calculation."
                
                return txt
                
            except Exception as e:
                return f"❌ Error in direct mapping: {str(e)}"
            
        else:  # Composite Inverse
            targets_to_check = [
                ("Elastic_Modulus", em_active, em_target),
                ("Poisson_X", px_active, px_target),
                ("Poisson_Y", py_active, py_target),
                ("Volume_Fraction", vf_active, vf_target)
            ]
            
            validation_errors = []
            for prop, is_active, target_val in targets_to_check:
                if is_active and target_val is not None:
                    has_data = any(
                        VALID_RANGES[s][o][prop] is not None 
                        for s in STRUCTURES 
                        for o in ORIENTATIONS
                    )
                    if not has_data:
                        validation_errors.append(f"{prop}: No training data available")
                        continue
                    
                    global_min = min(
                        VALID_RANGES[s][o][prop][0] 
                        for s in STRUCTURES 
                        for o in ORIENTATIONS 
                        if VALID_RANGES[s][o][prop] is not None
                    )
                    global_max = max(
                        VALID_RANGES[s][o][prop][1] 
                        for s in STRUCTURES 
                        for o in ORIENTATIONS 
                        if VALID_RANGES[s][o][prop] is not None
                    )
                    if not (global_min <= target_val <= global_max):
                        validation_errors.append(
                            f"{prop}={target_val:.{precision}f} outside global range [{global_min:.{precision}f}, {global_max:.{precision}f}]"
                        )
            
            if validation_errors:
                error_msg = "❌ Invalid target values:\n" + "\n".join(f"- {err}" for err in validation_errors)
                return error_msg
            
            return run_composite_inverse(
                em_active, em_target, em_tol,
                px_active, px_target, px_tol,
                py_active, py_target, py_tol,
                vf_active, vf_target, vf_tol,
                precision,
                selected_structures,
                selected_orientations
            )
            
    except Exception as e:
        return f"❌ Error: {str(e)}"

# -----------------------------------------------------------
# Gradio UI with Structure Selection and Direct Mapping
# -----------------------------------------------------------
with gr.Blocks(title="🧊 Forward and Inverse Design of Cellular Structures") as demo:
    gr.Markdown("# 🧊 Forward and Inverse Design of Cellular Structures")
    mode = gr.Radio(
        ["Forward", "Inverse", "Composite Inverse", "Direct Property Mapping"],
        value="Forward",
        label="Mode"
    )
    
    # Single precision control for ALL values
    precision_input = gr.Dropdown(
        choices=[1, 2, 3, 4, 5, 6],
        value=DEFAULT_PRECISION,
        label="🔢 Numerical Precision (Decimal Places)",
        info="Choose decimal places for ALL numerical values (thickness, Poisson's ratio, Elastic Modulus, Volume Fraction). This reduces duplicate results."
    )
    
    with gr.Row(visible=True) as row_fw_inv:
        structure_fi = gr.Dropdown(STRUCTURES, value="Reentrant", label="Structure")
        orientation_fi = gr.Dropdown(ORIENTATIONS, value="UVW", label="Orientation")
    thickness_input = gr.Number(value=1.5, label="Thickness (mm)", visible=True)
    
    with gr.Row(visible=False) as row_inv:
        target_prop = gr.Dropdown(PROPERTIES, value="Poisson_X", label="Target Property")
        target_val = gr.Number(value=-0.1, label="Target Value")
    
    # NEW: Direct Property Mapping UI
    with gr.Group(visible=False) as direct_mapping_group:
        gr.Markdown("### 🔁 Direct Property Mapping")
        gr.Markdown("Map directly from any input property to any output property")
        with gr.Row():
            with gr.Column():
                input_prop_direct = gr.Dropdown(
                    PROPERTIES, 
                    value="Volume_Fraction", 
                    label="Input Property"
                )
                input_val_direct = gr.Number(value=0.3, label="Input Value")
            with gr.Column():
                output_prop_direct = gr.Dropdown(
                    PROPERTIES, 
                    value="Elastic_Modulus", 
                    label="Output Property"
                )
                # Output value will be displayed in results
    
    with gr.Group(visible=False) as composite_group:
        gr.Markdown("### 🔧 Target Property Settings")
        with gr.Row():
            with gr.Column():
                em_active = gr.Checkbox(label="Activate Elastic Modulus", value=False)
                em_target = gr.Number(label="Target Value", value=100.0)
                em_tol = gr.Number(label="Tolerance", value=1.0)
            with gr.Column():
                px_active = gr.Checkbox(label="Activate Poisson_X", value=False)
                px_target = gr.Number(label="Target Value", value=-0.1)
                px_tol = gr.Number(label="Tolerance", value=0.01)
            with gr.Column():
                py_active = gr.Checkbox(label="Activate Poisson_Y", value=False)
                py_target = gr.Number(label="Target Value", value=-0.1)
                py_tol = gr.Number(label="Tolerance", value=0.01)
            with gr.Column():
                vf_active = gr.Checkbox(label="Activate Volume Fraction", value=False)
                vf_target = gr.Number(label="Target Value", value=0.3)
                vf_tol = gr.Number(label="Tolerance", value=0.01)
        
        # Structure and Orientation Selection
        gr.Markdown("### 🔍 Search Scope Selection")
        with gr.Row():
            with gr.Column():
                structure_selection = gr.CheckboxGroup(
                    choices=STRUCTURES,
                    value=STRUCTURES,
                    label="✅ Select Structures to Search",
                    info="Choose which structures to include in the search"
                )
            with gr.Column():
                orientation_selection = gr.CheckboxGroup(
                    choices=ORIENTATIONS,
                    value=ORIENTATIONS,
                    label="✅ Select Orientations to Search",
                    info="Choose which orientations to include in the search"
                )
    
    def toggle_visibility(mode):
        if mode == "Forward":
            return (
                gr.update(visible=True),   # row_fw_inv
                gr.update(visible=True),   # thickness_input
                gr.update(visible=False),  # row_inv
                gr.update(visible=False),  # direct_mapping_group
                gr.update(visible=False)   # composite_group
            )
        elif mode == "Inverse":
            return (
                gr.update(visible=True),
                gr.update(visible=False),
                gr.update(visible=True),
                gr.update(visible=False),
                gr.update(visible=False)
            )
        elif mode == "Direct Property Mapping":
            return (
                gr.update(visible=True),
                gr.update(visible=False),
                gr.update(visible=False),
                gr.update(visible=True),
                gr.update(visible=False)
            )
        else:  # Composite Inverse
            return (
                gr.update(visible=False),
                gr.update(visible=False),
                gr.update(visible=False),
                gr.update(visible=False),
                gr.update(visible=True)
            )
    
    mode.change(
        fn=toggle_visibility,
        inputs=mode,
        outputs=[
            row_fw_inv, 
            thickness_input, 
            row_inv, 
            direct_mapping_group, 
            composite_group
        ]
    )
    
    def update_range(s, o):
        b = get_thickness_bounds(s, o)
        if b is None:
            return gr.update(minimum=0.1, maximum=5.0, value=1.5)
        return gr.update(minimum=b[0], maximum=b[1], value=(b[0] + b[1]) / 2)
    
    structure_fi.change(fn=update_range, inputs=[structure_fi, orientation_fi], outputs=thickness_input)
    orientation_fi.change(fn=update_range, inputs=[structure_fi, orientation_fi], outputs=thickness_input)
    
    run_btn = gr.Button("Run", variant="primary")
    output = gr.Markdown("Results will appear here...")
    
    run_btn.click(
        fn=run_app,
        inputs=[
            mode,
            structure_fi, orientation_fi, thickness_input,
            target_prop, target_val,
            em_active, em_target, em_tol,
            px_active, px_target, px_tol,
            py_active, py_target, py_tol,
            vf_active, vf_target, vf_tol,
            precision_input,
            structure_selection,
            orientation_selection,
            input_prop_direct,  # NEW for Direct Mapping
            input_val_direct,   # NEW for Direct Mapping
            output_prop_direct  # NEW for Direct Mapping
        ],
        outputs=output
    )

if __name__ == "__main__":
    print("✅ Direct Property Mapping mode added")
    demo.launch(inbrowser=True, share=False)

✅ Valid parameter ranges computed from training data
✅ Dynamic normalization scales computed: {'Poisson_X': 1.7373999999999998, 'Poisson_Y': 1.7405, 'Elastic_Modulus': 1.007989722, 'Volume_Fraction': 0.8885}
✅ Direct Property Mapping mode added
Running on local URL:  http://127.0.0.1:7860

To create a public link, set `share=True` in `launch()`.


## AI Property Mapping Tool
This code provides an interactive Gradio-based user interface for analyzing and mapping relationships between the mechanical properties of lattice structures. By loading trained surrogate models, it enables accurate prediction of structural behavior. Using inverse design capabilities and numerical optimization, the tool can compute the optimal thickness required to achieve a target value of a given mechanical property (such as elastic modulus or Poisson’s ratio) and then map that thickness to another mechanical property, generating both the corresponding variation plots and numerical data. This process automatically extracts the valid ranges from the training data and prevents extrapolation beyond the model’s domain, ensuring that the design results remain physically and data-wise reliable.

In [1]:
import os
import numpy as np
import pandas as pd
import joblib
import gradio as gr
import matplotlib.pyplot as plt
from scipy.optimize import minimize_scalar

MODEL_DIR = "saved_models"
DATA_DIR = "training_datasets"

STRUCTURES = ["Reentrant","Gyroid","Diamond","Neovius","SchwarzP"]
ORIENTATIONS = ["UVW","UWV","VUW","VWU","WUV","WVU"]

PROPERTIES = [
    "Volume_Fraction",
    "Elastic_Modulus",
    "Poisson_X",
    "Poisson_Y"
]

OUTPUT_TO_KEY = {
    "Poisson_X": "px",
    "Poisson_Y": "py",
    "Elastic_Modulus": "em",
    "Volume_Fraction": "vf"
}

DEFAULT_PRECISION = 3

# ------------------------------------------------
# Utilities: compute VALID_RANGES from training CSVs
# ------------------------------------------------
def compute_valid_ranges():
    ranges = {}
    for structure in STRUCTURES:
        ranges[structure] = {}
        for orientation in ORIENTATIONS:
            struct_pattern = f"{structure.lower()}_{orientation.lower()}_train"
            files = [f for f in os.listdir(DATA_DIR)
                     if f.lower().endswith(".csv") and struct_pattern in f.lower()]
            if not files:
                ranges[structure][orientation] = {prop: None for prop in PROPERTIES}
                continue
            files.sort(key=lambda x: os.path.getmtime(os.path.join(DATA_DIR, x)), reverse=True)
            latest_file = files[0]
            try:
                df = pd.read_csv(os.path.join(DATA_DIR, latest_file))
                prop_ranges = {}
                for prop in PROPERTIES:
                    if prop in df.columns:
                        numeric_vals = pd.to_numeric(df[prop], errors="coerce").dropna()
                        if len(numeric_vals) > 0:
                            prop_ranges[prop] = (float(numeric_vals.min()), float(numeric_vals.max()))
                        else:
                            prop_ranges[prop] = None
                    else:
                        prop_ranges[prop] = None
                ranges[structure][orientation] = prop_ranges
            except Exception as e:
                print(f"[WARN] Error reading {latest_file}: {e}")
                ranges[structure][orientation] = {prop: None for prop in PROPERTIES}
    return ranges

VALID_RANGES = compute_valid_ranges()
print("[OK] VALID_RANGES computed")

# ------------------------------------------------
# Model Loader (robust to multiple candidates)
# ------------------------------------------------
def find_model_path(structure, orient, output):
    key = OUTPUT_TO_KEY[output]
    prefix = f"{structure.lower()}_{orient.lower()}_{key}"
    candidates = [f for f in os.listdir(MODEL_DIR)
                  if f.lower().startswith(prefix) and f.endswith(".joblib")]
    if not candidates:
        return None
    candidates.sort(key=lambda x: os.path.getmtime(os.path.join(MODEL_DIR, x)), reverse=True)
    return os.path.join(MODEL_DIR, candidates[0])

def load_model(structure, orient, output):
    path = find_model_path(structure, orient, output)
    if path is None:
        raise FileNotFoundError(f"No model found for {structure}-{orient}-{output}")
    return joblib.load(path)

# ------------------------------------------------
# Thickness bounds (robust, like Stage 4)
# ------------------------------------------------
def get_thickness_bounds(structure, orient):
    struct_lower = structure.lower()
    orient_lower = orient.lower()
    target_pattern = f"{struct_lower}_{orient_lower}_train"
    files = [f for f in os.listdir(DATA_DIR)
             if f.lower().endswith(".csv") and target_pattern in f.lower()]
    if not files:
        return None
    files.sort(key=lambda x: os.path.getmtime(os.path.join(DATA_DIR, x)), reverse=True)
    latest = files[0]
    try:
        df = pd.read_csv(os.path.join(DATA_DIR, latest))
        for c in df.columns:
            if "thick" in c.lower():
                vals = pd.to_numeric(df[c], errors="coerce").dropna().values
                if len(vals) > 0:
                    return float(np.min(vals)), float(np.max(vals))
    except Exception as e:
        print(f"[WARN] Failed to read bounds from {latest}: {e}")
    return None

# ------------------------------------------------
# Prediction (enhanced for Gyroid Poisson)
# ------------------------------------------------
def predict_property(structure, orient, output, thickness):
    bundle = load_model(structure, orient, output)
    X = np.array([[float(thickness)]])

    # symbolic + residual GPR
    algo = bundle.get("algo", "gpr")
    if algo == "symbolic_gpr":
        symbolic_model = bundle["symbolic_model"]
        sym_pred = symbolic_model.predict(X)[0]
        X_feat = np.column_stack([X, 1.0/(X+1e-6), np.log(X+1e-6)])
        X_scaled = bundle["scaler"].transform(X_feat)
        residual_pred = bundle["gpr"].predict(X_scaled)[0]
        return float(sym_pred + residual_pred)

    # Gyroid Poisson engineered features
    is_gyroid_poisson = bundle.get("is_gyroid_poisson", False)
    if is_gyroid_poisson:
        X_feat = np.column_stack([
            X,
            1.0 / (X + 1e-6),
            np.log(X + 1e-6),
            np.sqrt(X),
            X ** 0.333
        ])
        scaler = bundle["scaler"]
        Xs = scaler.transform(X_feat)
        gpr = bundle["gpr"]
        tree = bundle.get("tree", None)
        wg = bundle.get("wg", 1.0)
        wt = bundle.get("wt", 0.0)
        y_gpr = gpr.predict(Xs)[0]
        if tree is not None:
            y_tree = tree.predict(Xs)[0]
            y_final = wg * y_gpr + wt * y_tree
        else:
            y_final = y_gpr
        return float(y_final)

    # default poly + scaler + gpr (+ optional tree)
    poly = bundle.get("poly", None)
    scaler = bundle["scaler"]
    if poly is not None:
        Xf = poly.transform(X)
    else:
        # fallback to raw X if no poly in bundle
        Xf = X
    Xs = scaler.transform(Xf)
    gpr = bundle["gpr"]
    tree = bundle.get("tree", None)
    wg = bundle.get("wg", 1.0)
    wt = bundle.get("wt", 0.0)
    y_gpr = gpr.predict(Xs)[0]
    if tree is not None:
        y_tree = tree.predict(Xs)[0]
        y_final = wg * y_gpr + wt * y_tree
    else:
        y_final = y_gpr
    return float(y_final)

def predict_all_properties(structure, orient, thickness):
    results = {}
    for prop in PROPERTIES:
        try:
            bounds = get_thickness_bounds(structure, orient)
            if bounds and not (bounds[0] <= thickness <= bounds[1]):
                results[prop] = float('nan')
            else:
                results[prop] = predict_property(structure, orient, prop, thickness)
        except Exception as e:
            print(f"[WARN] predict_all_properties failed for {prop}: {e}")
            results[prop] = float('nan')
    return results

# ------------------------------------------------
# Inverse Design (single target)
# ------------------------------------------------
def inverse_design(structure, orient, prop, target_value, precision=DEFAULT_PRECISION, tol=1e-8, n_grid=800):
    vr = VALID_RANGES.get(structure, {}).get(orient, {})
    if not vr or vr.get(prop) is None:
        raise ValueError(f"No training data available for {prop} in {structure}-{orient}.")
    min_val, max_val = vr[prop]
    if not (min_val <= target_value <= max_val):
        raise ValueError(f"Target {target_value:.{precision}f} outside valid range [{min_val:.{precision}f}, {max_val:.{precision}f}] for {structure}-{orient}-{prop}")

    bounds = get_thickness_bounds(structure, orient)
    if bounds is None:
        raise RuntimeError("No thickness bounds")
    low, high = float(bounds[0]), float(bounds[1])

    best_err = float('inf')
    best_t = None
    grid = np.linspace(low, high, n_grid)

    for t in grid:
        try:
            pred = predict_property(structure, orient, prop, t)
            if not (min_val <= pred <= max_val):
                continue
            err = abs(pred - target_value)
            if err < best_err:
                best_err = err
                best_t = t
        except:
            continue

    if best_t is None:
        raise RuntimeError("No valid predictions found in grid search.")

    def objective(t):
        try:
            pred = predict_property(structure, orient, prop, t)
            if not (min_val <= pred <= max_val):
                return 1e10
            return (pred - target_value)**2
        except:
            return 1e10

    res = minimize_scalar(
        objective,
        bounds=(max(low, best_t - (high-low)/20), min(high, best_t + (high-low)/20)),
        method="bounded",
        options={"xatol": 10**(-precision-2), "maxiter": 300}
    )
    t_opt = float(res.x if res.success else best_t)
    return t_opt

# ------------------------------------------------
# Direct Property Mapping (from Stage 4)
# ------------------------------------------------
def direct_property_mapping(structure, orient, input_prop, input_value, output_prop, precision=DEFAULT_PRECISION):
    vr = VALID_RANGES.get(structure, {}).get(orient, {})
    if not vr or vr.get(input_prop) is None:
        raise ValueError(f"No training data available for {input_prop} in {structure}-{orient}.")
    in_min, in_max = vr[input_prop]
    if not (in_min <= input_value <= in_max):
        raise ValueError(f"Input {input_value:.{precision}f} outside valid range [{in_min:.{precision}f}, {in_max:.{precision}f}] for {structure}-{orient}-{input_prop}")
    if vr.get(output_prop) is None:
        raise ValueError(f"No training data available for {output_prop} in {structure}-{orient}.")

    bounds = get_thickness_bounds(structure, orient)
    if bounds is None:
        raise RuntimeError(f"No training data found for {structure}-{orient}.")
    low, high = float(bounds[0]), float(bounds[1])

    # coarse search
    grid_thicks = np.linspace(low, high, 500)
    best_err = float('inf')
    best_thick = None
    for t in grid_thicks:
        try:
            pred = predict_property(structure, orient, input_prop, t)
            if in_min <= pred <= in_max:
                err = abs(pred - input_value)
                if err < best_err:
                    best_err = err
                    best_thick = t
                    if err < 1e-8:
                        break
        except:
            continue
    if best_thick is None:
        raise RuntimeError(f"Could not find thickness for input {input_prop} = {input_value:.{precision}f}")

    # refine
    def objective(t):
        try:
            pred = predict_property(structure, orient, input_prop, t)
            if in_min <= pred <= in_max:
                return (pred - input_value)**2
        except:
            pass
        return 1e10

    res = minimize_scalar(
        objective,
        bounds=(low, high),
        method="bounded",
        options={"xatol": 10**(-precision-2), "maxiter": 200}
    )
    t_opt = float(res.x if res.success else best_thick)

    # output prediction
    out_val = predict_property(structure, orient, output_prop, t_opt)
    out_min, out_max = vr[output_prop]
    if not (out_min <= out_val <= out_max):
        raise RuntimeError(f"Predicted output {out_val:.{precision}f} outside valid range [{out_min:.{precision}f}, {out_max:.{precision}f}]")
    return t_opt, out_val

# ------------------------------------------------
# Batch Mapping
# ------------------------------------------------
def run_mapping(
    structure,
    orient,
    input_prop,
    output_prop,
    vmin,
    vmax,
    n_points
):
    precision = DEFAULT_PRECISION
    # Validate ranges using VALID_RANGES if موجود
    vr = VALID_RANGES.get(structure, {}).get(orient, {})
    if vr and vr.get(input_prop):
        in_min, in_max = vr[input_prop]
        # Clip user range into valid range (برای جلوگیری از empty)
        if vmin < in_min: vmin = in_min
        if vmax > in_max: vmax = in_max
        if vmin >= vmax:
            raise ValueError(f"Input range collapsed after validation: [{vmin:.{precision}f}, {vmax:.{precision}f}]")
    # Create input grid
    inputs = np.linspace(float(vmin), float(vmax), int(n_points))

    rows = []
    num_fail = 0

    # اگر حالت خاص Gyroid + VF -> Poisson_X/Y است، از Direct Property Mapping استفاده می‌کنیم
    use_direct = (structure == "Gyroid" and input_prop == "Volume_Fraction" and output_prop in ("Poisson_X","Poisson_Y"))

    for val in inputs:
        try:
            if use_direct:
                t_opt, out_val = direct_property_mapping(structure, orient, input_prop, float(val), output_prop, precision)
                rows.append([val, t_opt, out_val])
            else:
                # مسیر عمومی: وارونگی برای یافتن ضخامت نسبت به input_prop، سپس پیش‌بینی output_prop
                t_opt = inverse_design(structure, orient, input_prop, float(val), precision=precision)
                out_val = predict_property(structure, orient, output_prop, t_opt)
                rows.append([val, t_opt, out_val])
        except Exception as e:
            num_fail += 1
            # می‌توان لاگ چاپ کرد اما اجرای کلی را متوقف نکنیم
            # print(f"[SKIP] val={val}: {e}")
            continue

    if not rows:
        # در صورت عدم تولید داده، یک DataFrame خالی برمی‌گردانیم ولی با پیام واضح
        empty_df = pd.DataFrame(columns=[input_prop,"Thickness",output_prop])
        fig, ax = plt.subplots()
        ax.set_title(f"No valid data generated for {structure}-{orient}\n{output_prop} vs {input_prop}")
        ax.set_xlabel(input_prop)
        ax.set_ylabel(output_prop)
        ax.grid(True)
        return fig, empty_df, None

    df = pd.DataFrame(rows, columns=[input_prop,"Thickness",output_prop])

    # plot
    fig, ax = plt.subplots(figsize=(6,5), dpi=120)
    ax.plot(df[input_prop], df[output_prop], lw=2, color="#1f77b4")
    ax.set_xlabel(input_prop)
    ax.set_ylabel(output_prop)
    ax.set_title(f"{output_prop} vs {input_prop}\n{structure}-{orient}")
    ax.grid(True)

    # save CSV
    csv_path = "mapping_results.csv"
    try:
        df.to_csv(csv_path, index=False)
    except Exception as e:
        print(f"[WARN] Failed to save CSV: {e}")
        csv_path = None

    if num_fail > 0:
        print(f"[INFO] Skipped {num_fail} inputs due to out-of-range/model errors.")

    return fig, df, csv_path

# ------------------------------------------------
# UI
# ------------------------------------------------
with gr.Blocks() as demo:
    gr.Markdown("## AI Property Mapping Tool ")

    with gr.Row():
        structure = gr.Dropdown(
            STRUCTURES,
            value="Gyroid",
            label="Structure"
        )
        orient = gr.Dropdown(
            ORIENTATIONS,
            value="UVW",
            label="Orientation"
        )

    with gr.Row():
        input_prop = gr.Dropdown(
            PROPERTIES,
            value="Volume_Fraction",
            label="Input Property"
        )
        output_prop = gr.Dropdown(
            PROPERTIES,
            value="Elastic_Modulus",
            label="Output Property"
        )

    with gr.Row():
        vmin = gr.Number(
            value=0.1,
            label="Input Minimum"
        )
        vmax = gr.Number(
            value=0.4,
            label="Input Maximum"
        )
        n_points = gr.Slider(
            10,
            300,
            value=100,
            step=10,
            label="Number of Points"
        )

    run_btn = gr.Button("Run Mapping", variant="primary")
    plot = gr.Plot()
    table = gr.Dataframe()
    download = gr.File()

    # Optional: Auto-update vmin/vmax based on VALID_RANGES when user changes input_prop/structure/orientation
    def suggest_range(s, o, p):
        vr = VALID_RANGES.get(s, {}).get(o, {})
        if vr and vr.get(p):
            a, b = vr[p]
            # حاشیه کوچک برای راحتی
            span = b - a
            a2 = a + 0.02*span
            b2 = b - 0.02*span
            if a2 < b2:
                return gr.update(value=round(a2, 3)), gr.update(value=round(b2, 3))
            return gr.update(), gr.update()
        return gr.update(), gr.update()

    structure.change(fn=suggest_range, inputs=[structure, orient, input_prop], outputs=[vmin, vmax])
    orient.change(fn=suggest_range, inputs=[structure, orient, input_prop], outputs=[vmin, vmax])
    input_prop.change(fn=suggest_range, inputs=[structure, orient, input_prop], outputs=[vmin, vmax])

    run_btn.click(
        fn=run_mapping,
        inputs=[structure, orient, input_prop, output_prop, vmin, vmax, n_points],
        outputs=[plot, table, download]
    )

demo.launch()


[OK] VALID_RANGES computed
Running on local URL:  http://127.0.0.1:7860

To create a public link, set `share=True` in `launch()`.
